In [1]:

import pandas as pd

for res in ['square_002um', 'square_008um', 'square_016um']:
    df = pd.read_parquet(f'/project_antwerp/hbae/data/visium_hd_tonsil/binned_outputs/{res}/spatial/tissue_positions.parquet', engine='pyarrow')
    print(f'{res}: in_tissue={df.in_tissue.sum():,} / {len(df):,} = {df.in_tissue.mean():.3f}')


square_002um: in_tissue=11,210,791 / 11,222,500 = 0.999
square_008um: in_tissue=701,631 / 702,244 = 0.999
square_016um: in_tissue=175,448 / 175,561 = 0.999


In [1]:

import numpy as np
embs = np.load('/project_antwerp/hbae/Loki_output/0228_epoch10_finetune_embedding/fold_01/train_text_embs.npy')
print('train spot 수:', embs.shape[0])


train spot 수: 64157


In [2]:

import numpy as np
from anndata import read_h5ad

adata = read_h5ad('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_01/visium_hd_adata.h5ad')
X = adata.X.toarray()

print('val_exprs 통계:')
print(f'  shape: {X.shape}')
print(f'  mean: {X.mean():.4f}')
print(f'  zeros 비율: {(X==0).mean():.4f}')
print(f'  완전히 0인 bin 수: {(X.sum(axis=1)==0).sum():,}')
print(f'  완전히 0인 gene 수: {(X.sum(axis=0)==0).sum():,}')

# 임베딩 확인
embs = np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_01/visium_hd_img_embs.npy')
print()
print('임베딩 통계:')
print(f'  shape: {embs.shape}')
print(f'  mean: {embs.mean():.4f}')
print(f'  std: {embs.std():.4f}')
print(f'  norm (첫 5개): {np.linalg.norm(embs[:5], axis=1)}')


val_exprs 통계:
  shape: (701631, 2000)
  mean: 0.0323
  zeros 비율: 0.9570
  완전히 0인 bin 수: 417
  완전히 0인 gene 수: 0

임베딩 통계:
  shape: (701631, 768)
  mean: 0.0003
  std: 0.0361
  norm (첫 5개): [1.         1.         1.         1.0000001  0.99999994]


In [3]:

import numpy as np

exprs = np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_01/tile_exprs.npy')
print('shape:', exprs.shape)
print('mean:', exprs.mean())
print('max:', exprs.max())
print('min:', exprs.min())
print('zeros 비율:', (exprs==0).mean())
print('nan 수:', np.isnan(exprs).sum())
print('inf 수:', np.isinf(exprs).sum())


shape: (5618, 2000)
mean: 0.0
max: 0.0
min: 0.0
zeros 비율: 1.0
nan 수: 0
inf 수: 0


In [4]:

import numpy as np
import pandas as pd
import json

# bin 좌표 확인
scalef = 0.09202595
df = pd.read_parquet('/project_antwerp/hbae/data/visium_hd_tonsil/binned_outputs/square_008um/spatial/tissue_positions.parquet', engine='pyarrow')
df = df[df['in_tissue'] == 1]
df['hires_col'] = df['pxl_col_in_fullres'] * scalef
df['hires_row'] = df['pxl_row_in_fullres'] * scalef

print('bin hires_col 범위:', df.hires_col.min(), '~', df.hires_col.max())
print('bin hires_row 범위:', df.hires_row.min(), '~', df.hires_row.max())

# 타일 좌표 확인
coords = np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_01/tile_coords.npy')
print()
print('tile col 범위:', coords[:,0].min(), '~', coords[:,0].max())
print('tile row 범위:', coords[:,1].min(), '~', coords[:,1].max())
print('tile 크기 (stride):', coords[1,0] - coords[0,0])

# 첫 번째 타일에 bin이 있는지 확인
col_start, row_start = coords[0]
tile_size = 57
in_tile = (
    (df.hires_col >= col_start) & (df.hires_col < col_start + tile_size) &
    (df.hires_row >= row_start) & (df.hires_row < row_start + tile_size)
)
print()
print(f'첫 번째 타일 ({col_start}, {row_start}) 안의 bin 수:', in_tile.sum())


bin hires_col 범위: 2233.6568306790623 ~ 4494.42636711466
bin hires_row 범위: 659.7770928777305 ~ 2920.1191977022254

tile col 범위: 57 ~ 5130
tile row 범위: 0 ~ 5928
tile 크기 (stride): 57

첫 번째 타일 (2394, 0) 안의 bin 수: 0


In [1]:

import numpy as np
import torch
import torch.nn.functional as F

# 임베딩 유사도 분포 확인
val_embs   = torch.tensor(np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_01/tile_img_embs.npy')).float()
train_embs = torch.tensor(np.load('/project_antwerp/hbae/Loki_output/0228_embeddings_zeroshot/fold_01/train_text_embs.npy')).float()

val_norm   = F.normalize(val_embs, dim=-1)
train_norm = F.normalize(train_embs, dim=-1)

# 첫 번째 타일의 유사도 분포
sim = val_norm[0] @ train_norm.T
print('유사도 분포 (첫 번째 타일):')
print(f'  min: {sim.min():.4f}')
print(f'  max: {sim.max():.4f}')
print(f'  mean: {sim.mean():.4f}')
print(f'  neg 비율: {(sim<0).float().mean():.4f}')

# val_exprs 분포
exprs = np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_01/tile_exprs.npy')
print()
print('발현값 분포:')
print(f'  mean: {exprs.mean():.4f}')
print(f'  std: {exprs.std():.4f}')
print(f'  max: {exprs.max():.4f}')
print(f'  zeros: {(exprs==0).mean():.4f}')

# top 300 gene 발현값 확인
mean_expr = exprs.mean(axis=0)
top300_idx = np.argsort(mean_expr)[::-1][:300]
print()
print('top 300 gene 발현값:')
print(f'  mean: {exprs[:, top300_idx].mean():.4f}')
print(f'  zeros: {(exprs[:, top300_idx]==0).mean():.4f}')
print(f'  std across spots: {exprs[:, top300_idx].std(axis=0).mean():.4f}')


유사도 분포 (첫 번째 타일):
  min: 0.1207
  max: 0.4004
  mean: 0.2689
  neg 비율: 0.0000

발현값 분포:
  mean: 1.8828
  std: 1.5872
  max: 10.5558
  zeros: 0.2458

top 300 gene 발현값:
  mean: 4.2114
  zeros: 0.0074
  std across spots: 0.7962


In [2]:

import numpy as np

exprs     = np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_01/tile_exprs.npy')
gene_names = np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_01/tile_gene_names.npy', allow_pickle=True)

mean_expr  = exprs.mean(axis=0)
top300_idx = np.argsort(mean_expr)[::-1][:20]
print('Visium HD top 20 expressed genes:')
for i, idx in enumerate(top300_idx):
    print(f'  {i+1}. {gene_names[idx]}: {mean_expr[idx]:.3f}')

# HNSCC train exprs top genes 확인
train_exprs = np.load('/project_antwerp/hbae/Loki_output/0228_embeddings_zeroshot/fold_01/train_exprs.npy')
train_mean  = train_exprs.mean(axis=0)
train_top20 = np.argsort(train_mean)[::-1][:20]
print()
print('HNSCC train top 20 expressed genes (index):')
print(train_top20)
print('mean values:', train_mean[train_top20])


Visium HD top 20 expressed genes:
  1. IGKC: 8.617
  2. CD74: 7.883
  3. IGHG1: 7.143
  4. CORO1A: 6.673
  5. IGHA1: 6.481
  6. CLU: 6.478
  7. VIM: 6.434
  8. LTB: 6.120
  9. APOE: 5.890
  10. LYZ: 5.881
  11. H3F3A: 5.717
  12. CYBA: 5.697
  13. CXCR4: 5.683
  14. LSP1: 5.660
  15. CD37: 5.629
  16. CD44: 5.600
  17. LCP1: 5.525
  18. CRIP1: 5.470
  19. CCL19: 5.453
  20. AHNAK: 5.416

HNSCC train top 20 expressed genes (index):
[1569 1570  915 1034 1037 1370  312  401  402 1744  458  910  912  913
  405 1043 1381 1120 1956 1728]
mean values: [3.9133718 3.7615926 3.4090822 3.006123  2.7968895 2.454672  2.445838
 2.113067  2.0799026 2.0638146 2.0563982 2.0496185 1.9461159 1.9343442
 1.8658842 1.8638897 1.8399416 1.8384194 1.8107742 1.8070658]


In [3]:

import numpy as np
import torch
import torch.nn.functional as F

# Visium HD 임베딩
hd_embs = torch.tensor(np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_01/tile_img_embs.npy')).float()

# HNSCC val 임베딩 (fold_01)
hnscc_embs = torch.tensor(np.load('/project_antwerp/hbae/Loki_output/0228_epoch10_finetune_embedding/fold_01/val_img_embs.npy')).float()

hd_norm    = F.normalize(hd_embs, dim=-1)
hnscc_norm = F.normalize(hnscc_embs, dim=-1)

print('Visium HD 임베딩 분포:')
print(f'  mean: {hd_norm.mean():.4f}, std: {hd_norm.std():.4f}')

print('HNSCC val 임베딩 분포:')
print(f'  mean: {hnscc_norm.mean():.4f}, std: {hnscc_norm.std():.4f}')

# train text embs와의 유사도 비교
train_embs = torch.tensor(np.load('/project_antwerp/hbae/Loki_output/0228_epoch10_finetune_embedding/fold_01/train_text_embs.npy')).float()
train_norm = F.normalize(train_embs, dim=-1)

sim_hd    = (hd_norm @ train_norm.T)
sim_hnscc = (hnscc_norm @ train_norm.T)

print()
print('Visium HD → train 유사도:')
print(f'  mean: {sim_hd.mean():.4f}, std: {sim_hd.std():.4f}')
print(f'  min: {sim_hd.min():.4f}, max: {sim_hd.max():.4f}')

print()
print('HNSCC val → train 유사도:')
print(f'  mean: {sim_hnscc.mean():.4f}, std: {sim_hnscc.std():.4f}')
print(f'  min: {sim_hnscc.min():.4f}, max: {sim_hnscc.max():.4f}')


Visium HD 임베딩 분포:
  mean: 0.0003, std: 0.0361
HNSCC val 임베딩 분포:
  mean: 0.0006, std: 0.0361

Visium HD → train 유사도:
  mean: 0.2109, std: 0.0643
  min: -0.1250, max: 0.5225

HNSCC val → train 유사도:
  mean: 0.2163, std: 0.0705
  min: -0.1850, max: 0.5403


In [4]:

import numpy as np
import pandas as pd

scalef = 0.09202595
df = pd.read_parquet('/project_antwerp/hbae/data/visium_hd_tonsil/binned_outputs/square_008um/spatial/tissue_positions.parquet', engine='pyarrow')
df = df[df['in_tissue'] == 1]
df['hires_col'] = df['pxl_col_in_fullres'] * scalef
df['hires_row'] = df['pxl_row_in_fullres'] * scalef

bin_col = df['hires_col'].values
bin_row = df['hires_row'].values

coords = np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_01/tile_coords.npy')
tile_size = 57

# 첫 10개 타일의 bin 수 확인
print('타일별 bin 수 (첫 10개):')
for i in range(10):
    col_start, row_start = coords[i]
    in_tile = (
        (bin_col >= col_start) & (bin_col < col_start + tile_size) &
        (bin_row >= row_start) & (bin_row < row_start + tile_size)
    )
    print(f'  타일 ({col_start}, {row_start}): {in_tile.sum()}개 bin')


타일별 bin 수 (첫 10개):
  타일 (2233, 659): 378개 bin
  타일 (2290, 659): 396개 bin
  타일 (2347, 659): 378개 bin
  타일 (2404, 659): 378개 bin
  타일 (2461, 659): 378개 bin
  타일 (2518, 659): 378개 bin
  타일 (2575, 659): 402개 bin
  타일 (2632, 659): 399개 bin
  타일 (2689, 659): 399개 bin
  타일 (2746, 659): 399개 bin


In [ ]:

import numpy as np
import pandas as pd
import scanpy as sc

# count matrix 로드
adata = sc.read_10x_h5('/project_antwerp/hbae/data/visium_hd_tonsil/binned_outputs/square_008um/filtered_feature_bc_matrix.h5')
adata.var_names_make_unique()

print('count matrix 통계 (normalize 전):')
X = adata.X.toarray()
print(f'  shape: {X.shape}')
print(f'  mean: {X.mean():.4f}')
print(f'  max: {X.max():.4f}')
print(f'  zeros 비율: {(X==0).mean():.4f}')

# 첫 번째 타일 bin들의 count 확인
scalef = 0.09202595
df = pd.read_parquet('/project_antwerp/hbae/data/visium_hd_tonsil/binned_outputs/square_008um/spatial/tissue_positions.parquet', engine='pyarrow')
df = df[df['in_tissue'] == 1].reset_index(drop=True)
df['hires_col'] = df['pxl_col_in_fullres'] * scalef
df['hires_row'] = df['pxl_row_in_fullres'] * scalef

barcode_to_idx = {b: i for i, b in enumerate(adata.obs.index)}
df['count_idx'] = df['barcode'].map(barcode_to_idx)
df = df.dropna(subset=['count_idx'])
df['count_idx'] = df['count_idx'].astype(int)

bin_col = df['hires_col'].values
bin_row = df['hires_row'].values
bin_idx = df['count_idx'].values

# 첫 번째 타일 (2233, 659)
col_start, row_start = 2233, 659
tile_size = 57
in_tile = (
    (bin_col >= col_start) & (bin_col < col_start + tile_size) &
    (bin_row >= row_start) & (bin_row < row_start + tile_size)
)
tile_bin_idx = bin_idx[in_tile]
print(f'첫 번째 타일 bin 수: {len(tile_bin_idx)}')
tile_counts = X[tile_bin_idx]
print(f'tile count sum: {tile_counts.sum(axis=0)[:10]}')
print(f'tile count mean: {tile_counts.mean():.4f}')
print(f'tile count max: {tile_counts.max():.4f}')


/opt/conda/lib/python3.11/site-packages/anndata/_core/anndata.py:1798: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/opt/conda/lib/python3.11/site-packages/anndata/_core/anndata.py:1798: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


count matrix 통계 (normalize 전):
  shape: (701631, 18085)
  mean: 0.0301
  max: 2091.0000


In [1]:

import numpy as np
import pandas as pd
import scanpy as sc

# count matrix 로드 (toarray 없이 sparse로)
adata = sc.read_10x_h5('/project_antwerp/hbae/data/visium_hd_tonsil/binned_outputs/square_008um/filtered_feature_bc_matrix.h5')
adata.var_names_make_unique()

print('count matrix (sparse):')
print(f'  shape: {adata.X.shape}')
print(f'  nnz: {adata.X.nnz}')
print(f'  zeros 비율: {1 - adata.X.nnz / (adata.X.shape[0] * adata.X.shape[1]):.4f}')

# HVG 필터링 먼저
with open('/project_antwerp/hbae/data/0228_HVG_NEW/0228_HVG_Finetune_gene_list_full.txt') as f:
    hvg_genes = [line.strip() for line in f.readlines()]
adata = adata[:, adata.var_names.isin(hvg_genes)].copy()
print(f'  HVG 필터 후: {adata.X.shape}')

# 첫 번째 타일 bin들만 확인
scalef = 0.09202595
df = pd.read_parquet('/project_antwerp/hbae/data/visium_hd_tonsil/binned_outputs/square_008um/spatial/tissue_positions.parquet', engine='pyarrow')
df = df[df['in_tissue'] == 1].reset_index(drop=True)
df['hires_col'] = df['pxl_col_in_fullres'] * scalef
df['hires_row'] = df['pxl_row_in_fullres'] * scalef
barcode_to_idx = {b: i for i, b in enumerate(adata.obs.index)}
df['count_idx'] = df['barcode'].map(barcode_to_idx)
df = df.dropna(subset=['count_idx'])
df['count_idx'] = df['count_idx'].astype(int)

bin_col = df['hires_col'].values
bin_row = df['hires_row'].values
bin_idx = df['count_idx'].values

# 첫 번째 타일
col_start, row_start = 2233, 659
tile_size = 57
in_tile = (
    (bin_col >= col_start) & (bin_col < col_start + tile_size) &
    (bin_row >= row_start) & (bin_row < row_start + tile_size)
)
tile_bin_idx = bin_idx[in_tile]
print(f'첫 번째 타일 bin 수: {len(tile_bin_idx)}')

# sparse 슬라이싱
tile_counts = adata.X[tile_bin_idx]
tile_sum = np.array(tile_counts.sum(axis=0)).flatten()
print(f'tile count sum (top 5): {np.sort(tile_sum)[::-1][:5]}')
print(f'tile count sum mean: {tile_sum.mean():.4f}')
print(f'tile count sum max: {tile_sum.max():.4f}')
print(f'tile count zeros: {(tile_sum==0).mean():.4f}')


/opt/conda/lib/python3.11/site-packages/anndata/_core/anndata.py:1798: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/opt/conda/lib/python3.11/site-packages/anndata/_core/anndata.py:1798: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


count matrix (sparse):
  shape: (701631, 18085)
  nnz: 301586962
  zeros 비율: 0.9762
  HVG 필터 후: (701631, 2000)
첫 번째 타일 bin 수: 378
tile count sum (top 5): [15481.  8686.  2956.  2811.  2765.]
tile count sum mean: 68.4205
tile count sum max: 15481.0000
tile count zeros: 0.1435


In [2]:

import numpy as np

exprs = np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_01/tile_exprs.npy')
print('tile_exprs 통계:')
print(f'  shape: {exprs.shape}')
print(f'  mean: {exprs.mean():.4f}')
print(f'  max: {exprs.max():.4f}')
print(f'  zeros: {(exprs==0).mean():.4f}')
print(f'  nan: {np.isnan(exprs).sum()}')
print(f'  첫 번째 타일 top 5: {np.sort(exprs[0])[::-1][:5]}')


tile_exprs 통계:
  shape: (1600, 2000)
  mean: 1.8828
  max: 10.5558
  zeros: 0.2458
  nan: 0
  첫 번째 타일 top 5: [8.894374  8.316581  7.2391787 7.188919  7.172432 ]


In [3]:

import numpy as np

train_exprs = np.load('/project_antwerp/hbae/Loki_output/0228_epoch10_finetune_embedding/fold_07/train_exprs.npy')
gene_names = open('/project_antwerp/hbae/data/0228_HVG_NEW/0228_HVG_Finetune_gene_list_full.txt').read().strip().split('\n')

import pandas as pd
df = pd.DataFrame({'gene': gene_names, 'mean_expr': train_exprs.mean(axis=0)})

# IGKC, IGHG1 등 확인
targets = ['IGKC', 'IGHG1', 'IGHA1', 'IGHG3', 'CD74', 'LTB', 'CD37']
for t in targets:
    if t in df.gene.values:
        val = df[df.gene==t]['mean_expr'].values[0]
        rank = (df.mean_expr > val).sum() + 1
        print(f'{t}: mean={val:.4f}, rank={rank}위')
    else:
        print(f'{t}: HVG에 없음')


IGKC: mean=3.5256, rank=3위
IGHG1: mean=2.0462, rank=12위
IGHA1: mean=2.0737, rank=11위
IGHG3: mean=1.9675, rank=14위
CD74: mean=2.4487, rank=6위
LTB: mean=0.2254, rank=693위
CD37: mean=0.1802, rank=808위


In [5]:

import numpy as np
import torch
import torch.nn.functional as F

train_embs  = torch.tensor(np.load('/project_antwerp/hbae/Loki_output/0228_epoch10_finetune_embedding/fold_07/train_text_embs.npy')).float()
train_exprs = torch.tensor(np.load('/project_antwerp/hbae/Loki_output/0228_epoch10_finetune_embedding/fold_07/train_exprs.npy')).float()
val_embs    = torch.tensor(np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_07/tile_img_embs.npy')).float()
val_exprs   = np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_07/tile_exprs.npy')

train_norm = F.normalize(train_embs, dim=-1)
val_norm   = F.normalize(val_embs, dim=-1)

# 첫 번째 타일 예측
sim = val_norm[0] @ train_norm.T
top500_idx = sim.topk(500).indices
sim_k  = sim[top500_idx]
expr_k = train_exprs[top500_idx]
w = sim_k / sim_k.sum()
pred = (w[:, None] * expr_k).sum(dim=0).numpy()

gene_names = open('/project_antwerp/hbae/data/0228_HVG_NEW/0228_HVG_Finetune_gene_list_full.txt').read().strip().split('\n')

# 예측값 top 10
top10_pred = np.argsort(pred)[::-1][:10]
print('예측 top 10 genes:')
for i in top10_pred:
    print(f'  {gene_names[i]}: pred={pred[i]:.4f}, gt={val_exprs[0,i]:.4f}')

print()
# IGKC 확인
igkc_idx = gene_names.index('IGKC')
print(f'IGKC: pred={pred[igkc_idx]:.4f}, gt={val_exprs[0, igkc_idx]:.4f}')
cd74_idx = gene_names.index('CD74')
print(f'CD74: pred={pred[cd74_idx]:.4f}, gt={val_exprs[0, cd74_idx]:.4f}')


예측 top 10 genes:
  IGKC: pred=4.4667, gt=0.0000
  S100A9: pred=3.5521, gt=3.5113
  S100A8: pred=3.5488, gt=1.7420
  CD74: pred=2.8669, gt=0.0000
  IGHA1: pred=2.6393, gt=1.7420
  KRT16: pred=2.6008, gt=3.4971
  IGHG1: pred=2.5863, gt=1.3416
  COL1A2: pred=2.5358, gt=2.3877
  COL1A1: pred=2.4357, gt=4.2781
  KRT6B: pred=2.3493, gt=0.3859

IGKC: pred=4.4667, gt=0.0000
CD74: pred=2.8669, gt=0.0000


In [6]:

import numpy as np
import torch
import torch.nn.functional as F
from scipy.stats import pearsonr

train_embs  = torch.tensor(np.load('/project_antwerp/hbae/Loki_output/0228_epoch10_finetune_embedding/fold_07/train_text_embs.npy')).float()
train_exprs = torch.tensor(np.load('/project_antwerp/hbae/Loki_output/0228_epoch10_finetune_embedding/fold_07/train_exprs.npy')).float()
val_embs    = torch.tensor(np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_07/tile_img_embs.npy')).float()
val_exprs   = np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_07/tile_exprs.npy')

train_norm = F.normalize(train_embs, dim=-1)
val_norm   = F.normalize(val_embs, dim=-1)

gene_names = open('/project_antwerp/hbae/data/0228_HVG_NEW/0228_HVG_Finetune_gene_list_full.txt').read().strip().split('\n')

# IGKC gene-wise PCC 계산 (전체 타일)
igkc_idx = gene_names.index('IGKC')
cd74_idx = gene_names.index('CD74')

preds_igkc = []
preds_cd74 = []

for i in range(len(val_embs)):
    sim = val_norm[i] @ train_norm.T
    idx = sim.topk(500).indices
    w = sim[idx] / sim[idx].sum()
    pred = (w[:, None] * train_exprs[idx]).sum(dim=0).numpy()
    preds_igkc.append(pred[igkc_idx])
    preds_cd74.append(pred[cd74_idx])

preds_igkc = np.array(preds_igkc)
preds_cd74 = np.array(preds_cd74)
gt_igkc = val_exprs[:, igkc_idx]
gt_cd74 = val_exprs[:, cd74_idx]

print(f'IGKC - pred: mean={preds_igkc.mean():.3f}, std={preds_igkc.std():.3f}')
print(f'IGKC - gt:   mean={gt_igkc.mean():.3f}, std={gt_igkc.std():.3f}')
if gt_igkc.std() > 1e-8:
    r, _ = pearsonr(preds_igkc, gt_igkc)
    print(f'IGKC PCC: {r:.4f}')

print()
print(f'CD74 - pred: mean={preds_cd74.mean():.3f}, std={preds_cd74.std():.3f}')
print(f'CD74 - gt:   mean={gt_cd74.mean():.3f}, std={gt_cd74.std():.3f}')
if gt_cd74.std() > 1e-8:
    r, _ = pearsonr(preds_cd74, gt_cd74)
    print(f'CD74 PCC: {r:.4f}')


IGKC - pred: mean=3.895, std=0.633
IGKC - gt:   mean=0.116, std=0.293
IGKC PCC: 0.0049

CD74 - pred: mean=2.458, std=0.367
CD74 - gt:   mean=0.033, std=0.155
CD74 PCC: -0.0166


In [8]:

import numpy as np
import pandas as pd
import scanpy as sc
from anndata import AnnData
from scipy.sparse import csr_matrix

adata = sc.read_10x_h5('/project_antwerp/hbae/data/visium_hd_tonsil/binned_outputs/square_008um/filtered_feature_bc_matrix.h5')
adata.var_names_make_unique()
with open('/project_antwerp/hbae/data/0228_HVG_NEW/0228_HVG_Finetune_gene_list_full.txt') as f:
    hvg_genes = [line.strip() for line in f.readlines()]
adata = adata[:, adata.var_names.isin(hvg_genes)].copy()

gene_names = list(adata.var_names)
igkc_idx = gene_names.index('IGKC')

scalef = 0.09202595
df = pd.read_parquet('/project_antwerp/hbae/data/visium_hd_tonsil/binned_outputs/square_008um/spatial/tissue_positions.parquet', engine='pyarrow')
df = df[df['in_tissue'] == 1].reset_index(drop=True)
df['hires_col'] = df['pxl_col_in_fullres'] * scalef
df['hires_row'] = df['pxl_row_in_fullres'] * scalef
barcode_to_idx = {b: i for i, b in enumerate(adata.obs.index)}
df['count_idx'] = df['barcode'].map(barcode_to_idx)
df = df.dropna(subset=['count_idx'])
df['count_idx'] = df['count_idx'].astype(int)

bin_col = df['hires_col'].values
bin_row = df['hires_row'].values
bin_idx = df['count_idx'].values

# 중앙 타일
col_start, row_start = 3300, 1700
tile_size = 57
in_tile = (
    (bin_col >= col_start) & (bin_col < col_start + tile_size) &
    (bin_row >= row_start) & (bin_row < row_start + tile_size)
)
tile_bin_idx = bin_idx[in_tile]
print(f'중앙 타일 bin 수: {len(tile_bin_idx)}')

tile_counts = adata.X[tile_bin_idx]
tile_sum = np.array(tile_counts.sum(axis=0)).flatten()
print(f'IGKC count sum: {tile_sum[igkc_idx]:.1f}')
print(f'total count sum: {tile_sum.sum():.1f}')

# normalize 시뮬레이션
agg = AnnData(X=csr_matrix(tile_sum.reshape(1,-1)))
sc.pp.normalize_total(agg)
sc.pp.log1p(agg)
print(f'IGKC normalized: {agg.X.toarray()[0, igkc_idx]:.4f}')

# HNSCC train IGKC 분포
train_exprs = np.load('/project_antwerp/hbae/Loki_output/0228_epoch10_finetune_embedding/fold_07/train_exprs.npy')
print(f'HNSCC train IGKC mean: {train_exprs[:, igkc_idx].mean():.4f}')
print(f'HNSCC train IGKC std: {train_exprs[:, igkc_idx].std():.4f}')


/opt/conda/lib/python3.11/site-packages/anndata/_core/anndata.py:1798: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/opt/conda/lib/python3.11/site-packages/anndata/_core/anndata.py:1798: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


중앙 타일 bin 수: 454
IGKC count sum: 12589.0
total count sum: 26681.0
IGKC normalized: 9.4407
HNSCC train IGKC mean: 0.1875
HNSCC train IGKC std: 0.4382


In [9]:

import numpy as np
import torch
import torch.nn.functional as F

train_embs  = torch.tensor(np.load('/project_antwerp/hbae/Loki_output/0228_epoch10_finetune_embedding/fold_07/train_text_embs.npy')).float()
train_exprs = torch.tensor(np.load('/project_antwerp/hbae/Loki_output/0228_epoch10_finetune_embedding/fold_07/train_exprs.npy')).float()
val_embs    = torch.tensor(np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_07/tile_img_embs.npy')).float()

train_norm = F.normalize(train_embs, dim=-1)
val_norm   = F.normalize(val_embs, dim=-1)

# 전체 예측값 계산 (빠르게)
all_preds = []
for i in range(len(val_norm)):
    sim = val_norm[i] @ train_norm.T
    idx = sim.topk(500).indices
    w = sim[idx] / sim[idx].sum()
    pred = (w[:, None] * train_exprs[idx]).sum(dim=0).numpy()
    all_preds.append(pred)
all_preds = np.array(all_preds)

print('예측값 통계:')
print(f'  shape: {all_preds.shape}')
print(f'  mean: {all_preds.mean():.4f}')
print(f'  std across tiles (per gene): {all_preds.std(axis=0).mean():.4f}')
print(f'  std across genes (per tile): {all_preds.std(axis=1).mean():.4f}')
print()

# 타일마다 예측값이 다른가?
print('첫 5개 타일의 top-3 predicted genes:')
gene_names = open('/project_antwerp/hbae/data/0228_HVG_NEW/0228_HVG_Finetune_gene_list_full.txt').read().strip().split('\n')
for i in range(5):
    top3 = np.argsort(all_preds[i])[::-1][:3]
    print(f'  타일 {i}: {[gene_names[j] for j in top3]}')

# 예측값의 spatial variation
print()
print(f'예측값 std (tile간): {all_preds.std(axis=0).mean():.6f}')
val_exprs = np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_07/tile_exprs.npy')
print(f'실제값 std (tile간): {val_exprs.std(axis=0).mean():.6f}')


예측값 통계:
  shape: (1600, 2000)
  mean: 0.2922
  std across tiles (per gene): 0.0691
  std across genes (per tile): 0.4016

첫 5개 타일의 top-3 predicted genes:
  타일 0: ['IGKC', 'S100A9', 'S100A8']
  타일 1: ['IGKC', 'S100A8', 'S100A9']
  타일 2: ['IGKC', 'S100A9', 'S100A8']
  타일 3: ['IGKC', 'S100A9', 'S100A8']
  타일 4: ['S100A9', 'IGKC', 'S100A8']

예측값 std (tile간): 0.069061
실제값 std (tile간): 0.767394


In [10]:

import numpy as np
import torch
import torch.nn.functional as F

train_embs  = torch.tensor(np.load('/project_antwerp/hbae/Loki_output/0228_embeddings_zeroshot/fold_07/train_text_embs.npy')).float()
train_exprs = torch.tensor(np.load('/project_antwerp/hbae/Loki_output/0228_embeddings_zeroshot/fold_07/train_exprs.npy')).float()
val_embs    = torch.tensor(np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_07/tile_img_embs.npy')).float()

train_norm = F.normalize(train_embs, dim=-1)
val_norm   = F.normalize(val_embs, dim=-1)

all_preds = []
for i in range(len(val_norm)):
    sim = val_norm[i] @ train_norm.T
    idx = sim.topk(500).indices
    w = sim[idx] / sim[idx].sum()
    pred = (w[:, None] * train_exprs[idx]).sum(dim=0).numpy()
    all_preds.append(pred)
all_preds = np.array(all_preds)

print('Zero-shot 예측값 통계:')
print(f'  std across tiles (per gene): {all_preds.std(axis=0).mean():.6f}')

gene_names = open('/project_antwerp/hbae/data/0228_HVG_NEW/0228_HVG_Finetune_gene_list_full.txt').read().strip().split('\n')
print('첫 5개 타일의 top-3 predicted genes:')
for i in range(5):
    top3 = np.argsort(all_preds[i])[::-1][:3]
    print(f'  타일 {i}: {[gene_names[j] for j in top3]}')

val_exprs = np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_07/tile_exprs.npy')
print(f'실제값 std (tile간): {val_exprs.std(axis=0).mean():.6f}')


Zero-shot 예측값 통계:
  std across tiles (per gene): 0.067399
첫 5개 타일의 top-3 predicted genes:
  타일 0: ['IGKC', 'S100A8', 'S100A9']
  타일 1: ['IGKC', 'S100A8', 'S100A9']
  타일 2: ['IGKC', 'S100A8', 'S100A9']
  타일 3: ['IGKC', 'S100A8', 'S100A9']
  타일 4: ['IGKC', 'S100A8', 'S100A9']
실제값 std (tile간): 0.767394


In [11]:

import numpy as np
import torch
import torch.nn.functional as F

train_embs = torch.tensor(np.load('/project_antwerp/hbae/Loki_output/0228_embeddings_zeroshot/fold_07/train_text_embs.npy')).float()
val_embs   = torch.tensor(np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_07/tile_img_embs.npy')).float()

train_norm = F.normalize(train_embs, dim=-1)
val_norm   = F.normalize(val_embs, dim=-1)

# 각 타일의 유사도 분포 확인
print('타일별 유사도 분포 (첫 5개):')
for i in range(5):
    sim = val_norm[i] @ train_norm.T
    print(f'  타일 {i}: min={sim.min():.4f}, max={sim.max():.4f}, mean={sim.mean():.4f}, std={sim.std():.4f}')

# 타일간 임베딩 유사도 (Visium HD 타일끼리)
sim_between = val_norm @ val_norm.T
print()
print(f'Visium HD 타일간 유사도:')
print(f'  mean: {sim_between.mean():.4f}')
print(f'  std:  {sim_between.std():.4f}')

# HNSCC val 타일간 유사도
hnscc_val = torch.tensor(np.load('/project_antwerp/hbae/Loki_output/0228_embeddings_zeroshot/fold_07/val_img_embs.npy')).float()
hnscc_norm = F.normalize(hnscc_val, dim=-1)
sim_hnscc = hnscc_norm @ hnscc_norm.T
print()
print(f'HNSCC val 타일간 유사도:')
print(f'  mean: {sim_hnscc.mean():.4f}')
print(f'  std:  {sim_hnscc.std():.4f}')


타일별 유사도 분포 (첫 5개):
  타일 0: min=0.1325, max=0.3822, mean=0.2713, std=0.0347
  타일 1: min=0.1328, max=0.3696, mean=0.2661, std=0.0315
  타일 2: min=0.1614, max=0.3873, mean=0.2839, std=0.0309
  타일 3: min=0.1679, max=0.4021, mean=0.2910, std=0.0309
  타일 4: min=0.1518, max=0.3978, mean=0.2814, std=0.0322

Visium HD 타일간 유사도:
  mean: 0.7361
  std:  0.1428

HNSCC val 타일간 유사도:
  mean: 0.6361
  std:  0.1331


In [12]:

import numpy as np
import scanpy as sc

# tile_exprs 유전자 순서
gene_names_hd = np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_07/tile_gene_names.npy', allow_pickle=True)

# train_exprs 유전자 순서
gene_names_train = open('/project_antwerp/hbae/data/0228_HVG_NEW/0228_HVG_Finetune_gene_list_full.txt').read().strip().split('\n')

print('tile_exprs 유전자 수:', len(gene_names_hd))
print('train_exprs 유전자 수:', len(gene_names_train))
print()
print('tile_exprs 첫 5개:', gene_names_hd[:5])
print('train_exprs 첫 5개:', gene_names_train[:5])
print()

# 순서가 같은지 확인
match = all(a == b for a, b in zip(gene_names_hd, gene_names_train))
print('유전자 순서 일치:', match)

# 불일치 위치 찾기
if not match:
    for i, (a, b) in enumerate(zip(gene_names_hd, gene_names_train)):
        if a != b:
            print(f'첫 불일치 위치 {i}: hd={a}, train={b}')
            break


tile_exprs 유전자 수: 2000
train_exprs 유전자 수: 2000

tile_exprs 첫 5개: ['ISG15' 'TNFRSF18' 'TNFRSF4' 'SCNN1D' 'MXRA8']
train_exprs 첫 5개: ['A2M', 'A2ML1', 'AATK', 'ABCA3', 'ABCA7']

유전자 순서 일치: False
첫 불일치 위치 0: hd=ISG15, train=A2M


In [13]:

import numpy as np

gene_names_hd    = np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_07/tile_gene_names.npy', allow_pickle=True)
gene_names_train = open('/project_antwerp/hbae/data/0228_HVG_NEW/0228_HVG_Finetune_gene_list_full.txt').read().strip().split('\n')

# 공통 유전자 확인
hd_set    = set(gene_names_hd)
train_set = set(gene_names_train)
common    = hd_set & train_set
print(f'공통 유전자: {len(common)} / HD:{len(hd_set)} / train:{len(train_set)}')

# tile_exprs를 train 순서로 재정렬
tile_exprs = np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_07/tile_exprs.npy')
hd_gene_to_idx = {g: i for i, g in enumerate(gene_names_hd)}

# train 순서 기준으로 재정렬
new_exprs = np.zeros((tile_exprs.shape[0], len(gene_names_train)), dtype=np.float32)
for j, gene in enumerate(gene_names_train):
    if gene in hd_gene_to_idx:
        new_exprs[:, j] = tile_exprs[:, hd_gene_to_idx[gene]]

print(f'재정렬 후 shape: {new_exprs.shape}')
print(f'zeros 비율: {(new_exprs==0).mean():.4f}')

# 저장
np.save('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_07/tile_exprs_reordered.npy', new_exprs)
print('저장 완료: tile_exprs_reordered.npy')


공통 유전자: 2000 / HD:2000 / train:2000
재정렬 후 shape: (1600, 2000)
zeros 비율: 0.2458
저장 완료: tile_exprs_reordered.npy


In [14]:

import numpy as np

# 재정렬 후 첫 5개 유전자 확인
gene_names_train = open('/project_antwerp/hbae/data/0228_HVG_NEW/0228_HVG_Finetune_gene_list_full.txt').read().strip().split('\n')
tile_exprs = np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_07/tile_exprs_reordered.npy')

print('재정렬 후 첫 5개 유전자:', gene_names_train[:5])
print('해당 발현값 (첫 타일):', tile_exprs[0, :5])

# IGKC 확인
igkc_idx = gene_names_train.index('IGKC')
print(f'IGKC (index {igkc_idx}) 발현값 mean: {tile_exprs[:, igkc_idx].mean():.4f}')


재정렬 후 첫 5개 유전자: ['A2M', 'A2ML1', 'AATK', 'ABCA3', 'ABCA7']
해당 발현값 (첫 타일): [3.4383948 0.3858674 1.6559409 1.7420256 4.0100985]
IGKC (index 915) 발현값 mean: 8.6167


In [15]:
# fold_01 tile_exprs도 확인

import numpy as np
gene_names_hd    = np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_01/tile_gene_names.npy', allow_pickle=True)
gene_names_train = open('/project_antwerp/hbae/data/0228_HVG_NEW/0228_HVG_Finetune_gene_list_full.txt').read().strip().split('\n')
print('fold_01 첫 5개:', gene_names_hd[:5])
print('train 첫 5개:', gene_names_train[:5])
print('순서 일치:', all(a==b for a,b in zip(gene_names_hd, gene_names_train)))


fold_01 첫 5개: ['ISG15' 'TNFRSF18' 'TNFRSF4' 'SCNN1D' 'MXRA8']
train 첫 5개: ['A2M', 'A2ML1', 'AATK', 'ABCA3', 'ABCA7']
순서 일치: False


In [16]:

import numpy as np
import pandas as pd

scalef = 0.09202595
df = pd.read_parquet('/project_antwerp/hbae/data/visium_hd_tonsil/binned_outputs/square_008um/spatial/tissue_positions.parquet', engine='pyarrow')
df = df[df['in_tissue'] == 1].reset_index(drop=True)
df['hires_col'] = df['pxl_col_in_fullres'] * scalef
df['hires_row'] = df['pxl_row_in_fullres'] * scalef

bin_col = df['hires_col'].values
bin_row = df['hires_row'].values

coords = np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_07/tile_coords.npy')
tile_size = 57

# 타일별 bin 수 분포 확인
bin_counts = []
for col_start, row_start in coords:
    in_tile = (
        (bin_col >= col_start) & (bin_col < col_start + tile_size) &
        (bin_row >= row_start) & (bin_row < row_start + tile_size)
    )
    bin_counts.append(in_tile.sum())

bin_counts = np.array(bin_counts)
print(f'타일당 bin 수 분포:')
print(f'  min: {bin_counts.min()}')
print(f'  max: {bin_counts.max()}')
print(f'  mean: {bin_counts.mean():.1f}')
print(f'  median: {np.median(bin_counts):.1f}')
print()
print(f'bin 10개 미만 타일: {(bin_counts < 10).sum()}개')
print(f'bin 50개 미만 타일: {(bin_counts < 50).sum()}개')
print(f'bin 100개 미만 타일: {(bin_counts < 100).sum()}개')
print(f'bin 300개 이상 타일: {(bin_counts >= 300).sum()}개')


타일당 bin 수 분포:
  min: 121
  max: 484
  mean: 438.5
  median: 441.0

bin 10개 미만 타일: 0개
bin 50개 미만 타일: 0개
bin 100개 미만 타일: 0개
bin 300개 이상 타일: 1528개


In [17]:
# fold_01 tile_exprs 재정렬

import numpy as np

gene_names_hd    = np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_01/tile_gene_names.npy', allow_pickle=True)
gene_names_train = open('/project_antwerp/hbae/data/0228_HVG_NEW/0228_HVG_Finetune_gene_list_full.txt').read().strip().split('\n')
tile_exprs       = np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_01/tile_exprs.npy')

hd_gene_to_idx = {g: i for i, g in enumerate(gene_names_hd)}
new_exprs = np.zeros((tile_exprs.shape[0], len(gene_names_train)), dtype=np.float32)
for j, gene in enumerate(gene_names_train):
    if gene in hd_gene_to_idx:
        new_exprs[:, j] = tile_exprs[:, hd_gene_to_idx[gene]]

np.save('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_01/tile_exprs.npy', new_exprs)
print('fold_01 재정렬 완료:', new_exprs.shape)


fold_01 재정렬 완료: (1600, 2000)


In [20]:

import numpy as np
import torch
import torch.nn.functional as F
from scipy.stats import pearsonr

train_embs  = torch.tensor(np.load('/project_antwerp/hbae/Loki_output/0228_epoch10_finetune_embedding/fold_01/train_text_embs.npy')).float()
train_exprs = torch.tensor(np.load('/project_antwerp/hbae/Loki_output/0228_epoch10_finetune_embedding/fold_01/train_exprs.npy')).float()
val_embs    = torch.tensor(np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_01/tile_img_embs.npy')).float()
val_exprs   = np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_01/tile_exprs.npy')

train_norm = F.normalize(train_embs, dim=-1)
val_norm   = F.normalize(val_embs, dim=-1)

gene_names = open('/project_antwerp/hbae/data/0228_HVG_NEW/0228_HVG_Finetune_gene_list_full.txt').read().strip().split('\n')

preds = []
for i in range(len(val_norm)):
    sim = val_norm[i] @ train_norm.T
    idx = sim.topk(500).indices
    w   = sim[idx] / sim[idx].sum()
    preds.append((w[:, None] * train_exprs[idx]).sum(0).numpy())
preds = np.array(preds)

mean_expr  = val_exprs.mean(axis=0)
top300_idx = np.argsort(mean_expr)[::-1][:300]

gene_pccs = []
for g in top300_idx:
    if val_exprs[:, g].std() > 1e-8:
        r, _ = pearsonr(preds[:, g], val_exprs[:, g])
        if np.isfinite(r):
            gene_pccs.append((gene_names[g], r, val_exprs[:, g].mean(), preds[:, g].mean()))

gene_pccs.sort(key=lambda x: -x[1])
print('Gene-wise PCC 상위 10개:')
print('gene            pcc       gt_mean   pred_mean')
print('-'*55)
for g, r, gt, pred in gene_pccs[:10]:
    print(g.ljust(15), round(r,4), round(gt,4), round(pred,4))

print()
print('Gene-wise PCC 하위 10개:')
for g, r, gt, pred in gene_pccs[-10:]:
    print(g.ljust(15), round(r,4), round(gt,4), round(pred,4))

print()
print('전체 gene-wise PCC 분포:')
all_r = [x[1] for x in gene_pccs]
print('mean:', round(np.mean(all_r),4))
print('median:', round(np.median(all_r),4))
print('std:', round(np.std(all_r),4))
print('양수 비율:', round((np.array(all_r)>0).mean(),4))


Gene-wise PCC 상위 10개:
gene            pcc       gt_mean   pred_mean
-------------------------------------------------------
PTGDS           0.1774 5.325 0.588
C3              0.167 5.0425 0.9047
CXCL14          0.1653 4.3962 1.2161
ZAP70           0.1517 4.1573 0.1445
TXNIP           0.1454 5.0562 1.1505
CD8A            0.1446 3.6746 0.1222
ABI3BP          0.1406 3.5459 0.1848
TSPAN4          0.1405 3.4274 0.3536
CD6             0.1387 4.206 0.1183
TRAF3IP3        0.1354 4.3418 0.1276

Gene-wise PCC 하위 10개:
CYBA            -0.0707 5.6966 0.8153
TNFAIP3         -0.0711 3.9405 0.2553
TNFAIP2         -0.0726 3.8397 0.4932
AHNAK           -0.0744 5.4163 1.4241
CD79A           -0.0757 4.9824 0.2635
BST2            -0.0823 4.21 1.0811
IGLC1           -0.0852 3.5005 2.0826
COL4A2          -0.088 4.2264 0.6417
ALOX5AP         -0.0915 3.6962 0.179
ACTN1           -0.1553 3.5664 0.7779

전체 gene-wise PCC 분포:
mean: 0.0397
median: 0.0439
std: 0.0565
양수 비율: 0.7567


In [21]:

import numpy as np
import torch
import torch.nn.functional as F

val_embs   = torch.tensor(np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_01/tile_img_embs.npy')).float()
hnscc_embs = torch.tensor(np.load('/project_antwerp/hbae/Loki_output/0228_epoch10_finetune_embedding/fold_01/val_img_embs.npy')).float()

val_norm   = F.normalize(val_embs, dim=-1)
hnscc_norm = F.normalize(hnscc_embs, dim=-1)

# 타일간 유사도 분포
sim_hd    = (val_norm @ val_norm.T).numpy()
sim_hnscc = (hnscc_norm @ hnscc_norm.T).numpy()

# 대각선 제외
np.fill_diagonal(sim_hd, 0)
np.fill_diagonal(sim_hnscc, 0)

n_hd    = len(val_norm)
n_hnscc = len(hnscc_norm)

print('Visium HD 타일간 유사도 (자기자신 제외):')
print(f'  mean: {sim_hd.sum()/(n_hd*(n_hd-1)):.4f}')
print(f'  std:  {sim_hd[sim_hd!=0].std():.4f}')

print()
print('HNSCC val 타일간 유사도 (자기자신 제외):')
print(f'  mean: {sim_hnscc.sum()/(n_hnscc*(n_hnscc-1)):.4f}')
print(f'  std:  {sim_hnscc[sim_hnscc!=0].std():.4f}')


Visium HD 타일간 유사도 (자기자신 제외):
  mean: 0.7582
  std:  0.1438

HNSCC val 타일간 유사도 (자기자신 제외):
  mean: 0.3261
  std:  0.1562


In [22]:

import numpy as np

# train_exprs 분포 확인
train_exprs = np.load('/project_antwerp/hbae/Loki_output/0228_epoch10_finetune_embedding/fold_01/train_exprs.npy')
val_exprs   = np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_01/tile_exprs.npy')

print('HNSCC train_exprs:')
print(f'  mean: {train_exprs.mean():.4f}')
print(f'  std:  {train_exprs.std():.4f}')
print(f'  max:  {train_exprs.max():.4f}')
print(f'  zeros 비율: {(train_exprs==0).mean():.4f}')

print()
print('Visium HD tile_exprs:')
print(f'  mean: {val_exprs.mean():.4f}')
print(f'  std:  {val_exprs.std():.4f}')
print(f'  max:  {val_exprs.max():.4f}')
print(f'  zeros 비율: {(val_exprs==0).mean():.4f}')


HNSCC train_exprs:
  mean: 0.2906
  std:  0.6365
  max:  8.2888
  zeros 비율: 0.7434

Visium HD tile_exprs:
  mean: 1.8828
  std:  1.5872
  max:  10.5558
  zeros 비율: 0.2458


In [23]:

import numpy as np

exprs = np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_01/tile_exprs.npy')
train = np.load('/project_antwerp/hbae/Loki_output/0228_epoch10_finetune_embedding/fold_01/train_exprs.npy')

print('tile_exprs:')
print(f'  mean: {exprs.mean():.4f}')
print(f'  std:  {exprs.std():.4f}')
print(f'  max:  {exprs.max():.4f}')
print(f'  zeros: {(exprs==0).mean():.4f}')
print(f'  std (tile간, per gene): {exprs.std(axis=0).mean():.4f}')

print()
print('train_exprs:')
print(f'  mean: {train.mean():.4f}')
print(f'  std:  {train.std():.4f}')
print(f'  std (spot간, per gene): {train.std(axis=0).mean():.4f}')


tile_exprs:
  mean: 0.0323
  std:  0.1030
  max:  4.2285
  zeros: 0.2458
  std (tile간, per gene): 0.0235

train_exprs:
  mean: 0.2906
  std:  0.6365
  std (spot간, per gene): 0.4383


In [1]:

import numpy as np
from scipy.stats import pearsonr

# 간단한 예시
real = np.array([8.5, 8.6, 8.4, 8.5, 8.7, 8.3, 8.6, 8.5])
pred = np.array([3.8, 3.9, 3.8, 3.9, 4.0, 3.7, 3.9, 3.8])

r, _ = pearsonr(real, pred)
print(f'magnitude 달라도 패턴 같으면 PCC: {r:.4f}')

# 실제 IGKC 확인
gene_names = open('/project_antwerp/hbae/data/0228_HVG_NEW/0228_HVG_Finetune_gene_list_full.txt').read().strip().split('\n')
igkc_idx = gene_names.index('IGKC')

exprs = np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_01/tile_exprs.npy')
print(f'IGKC 실제값 std: {exprs[:, igkc_idx].std():.4f}')
print(f'IGKC 실제값 범위: {exprs[:, igkc_idx].min():.4f} ~ {exprs[:, igkc_idx].max():.4f}')


magnitude 달라도 패턴 같으면 PCC: 0.9285
IGKC 실제값 std: 0.8769
IGKC 실제값 범위: 0.0816 ~ 4.2285


In [2]:

import numpy as np
import torch
import torch.nn.functional as F
from scipy.stats import pearsonr

train_embs  = torch.tensor(np.load('/project_antwerp/hbae/Loki_output/0228_epoch10_finetune_embedding/fold_01/train_text_embs.npy')).float()
train_exprs = torch.tensor(np.load('/project_antwerp/hbae/Loki_output/0228_epoch10_finetune_embedding/fold_01/train_exprs.npy')).float()
val_embs    = torch.tensor(np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_01/tile_img_embs.npy')).float()
val_exprs   = np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_01/tile_exprs.npy')

train_norm = F.normalize(train_embs, dim=-1)
val_norm   = F.normalize(val_embs, dim=-1)

gene_names = open('/project_antwerp/hbae/data/0228_HVG_NEW/0228_HVG_Finetune_gene_list_full.txt').read().strip().split('\n')
igkc_idx   = gene_names.index('IGKC')

# 전체 예측
preds = []
for i in range(len(val_norm)):
    sim = val_norm[i] @ train_norm.T
    idx = sim.topk(500).indices
    w   = sim[idx] / sim[idx].sum()
    preds.append((w[:, None] * train_exprs[idx]).sum(0).numpy())
preds = np.array(preds)

print('IGKC 예측값 std:', preds[:, igkc_idx].std())
print('IGKC 예측값 범위:', preds[:, igkc_idx].min(), '~', preds[:, igkc_idx].max())
print('IGKC 실제값 std:', val_exprs[:, igkc_idx].std())
print('IGKC 실제값 범위:', val_exprs[:, igkc_idx].min(), '~', val_exprs[:, igkc_idx].max())

r, _ = pearsonr(preds[:, igkc_idx], val_exprs[:, igkc_idx])
print(f'IGKC Gene-wise PCC: {r:.4f}')

# top 20 gene PCC
mean_expr = val_exprs.mean(axis=0)
top20_idx = np.argsort(mean_expr)[::-1][:20]
print()
print('Top 20 genes PCC:')
for idx in top20_idx:
    if val_exprs[:, idx].std() > 1e-8:
        r, _ = pearsonr(preds[:, idx], val_exprs[:, idx])
        print(f'  {gene_names[idx]}: pred_std={preds[:,idx].std():.3f}, gt_std={val_exprs[:,idx].std():.3f}, PCC={r:.4f}')


IGKC 예측값 std: 0.78295934
IGKC 예측값 범위: 1.7051675 ~ 5.53356
IGKC 실제값 std: 0.87686354
IGKC 실제값 범위: 0.08158713 ~ 4.2285285
IGKC Gene-wise PCC: -0.0063

Top 20 genes PCC:
  IGKC: pred_std=0.783, gt_std=0.877, PCC=-0.0063
  CD74: pred_std=0.309, gt_std=0.507, PCC=-0.0322
  IGHG1: pred_std=0.700, gt_std=0.599, PCC=-0.0414
  CORO1A: pred_std=0.150, gt_std=0.395, PCC=0.0347
  CLU: pred_std=0.338, gt_std=0.411, PCC=0.0152
  VIM: pred_std=0.201, gt_std=0.280, PCC=0.0302
  IGHA1: pred_std=0.619, gt_std=0.524, PCC=-0.0808
  LTB: pred_std=0.104, gt_std=0.273, PCC=0.0055
  CCL19: pred_std=0.265, gt_std=0.385, PCC=0.1051
  APOE: pred_std=0.152, gt_std=0.250, PCC=-0.0495
  CD37: pred_std=0.099, gt_std=0.257, PCC=-0.0121
  LYZ: pred_std=0.200, gt_std=0.199, PCC=0.0275
  CXCR4: pred_std=0.152, gt_std=0.189, PCC=0.0211
  CCL21: pred_std=0.121, gt_std=0.420, PCC=0.0717
  H3F3A: pred_std=0.194, gt_std=0.176, PCC=-0.0576
  CYBA: pred_std=0.124, gt_std=0.145, PCC=-0.0791
  S100A9: pred_std=0.475, gt_std=0.582

In [3]:

import numpy as np
import torch
import torch.nn.functional as F

train_embs  = torch.tensor(np.load('/project_antwerp/hbae/Loki_output/0228_epoch10_finetune_embedding/fold_01/train_text_embs.npy')).float()
train_exprs = torch.tensor(np.load('/project_antwerp/hbae/Loki_output/0228_epoch10_finetune_embedding/fold_01/train_exprs.npy')).float()
val_embs    = torch.tensor(np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_01/tile_img_embs.npy')).float()
val_exprs   = np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_01/tile_exprs.npy')
coords      = np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_01/tile_coords.npy')

train_norm = F.normalize(train_embs, dim=-1)
val_norm   = F.normalize(val_embs, dim=-1)

gene_names = open('/project_antwerp/hbae/data/0228_HVG_NEW/0228_HVG_Finetune_gene_list_full.txt').read().strip().split('\n')
igkc_idx   = gene_names.index('IGKC')

preds = []
for i in range(len(val_norm)):
    sim = val_norm[i] @ train_norm.T
    idx = sim.topk(500).indices
    w   = sim[idx] / sim[idx].sum()
    preds.append((w[:, None] * train_exprs[idx]).sum(0).numpy())
preds = np.array(preds)

# 공간적 패턴 확인 (col 기준 정렬)
sort_idx = np.argsort(coords[:, 0])
print('col 순서로 정렬한 IGKC (첫 10개):')
print('col    pred    gt')
for i in sort_idx[:10]:
    print(f'{coords[i,0]:5.0f}  {preds[i,igkc_idx]:.3f}  {val_exprs[i,igkc_idx]:.3f}')


col 순서로 정렬한 IGKC (첫 10개):
col    pred    gt
 2233  4.924  1.915
 2233  4.817  3.804
 2233  4.829  3.387
 2233  4.668  3.954
 2233  4.958  2.511
 2233  4.936  1.682
 2233  3.729  3.860
 2233  5.158  3.400
 2233  5.021  3.731
 2233  4.847  3.865


In [4]:

import numpy as np
import pandas as pd
import scanpy as sc
import torch
import torch.nn.functional as F
from scipy.stats import pearsonr

print('='*60)
print('파이프라인 검증')
print('='*60)

# ── 1. 유전자 순서 확인
gene_names_hd    = np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_01/tile_gene_names.npy', allow_pickle=True)
gene_names_train = open('/project_antwerp/hbae/data/0228_HVG_NEW/0228_HVG_Finetune_gene_list_full.txt').read().strip().split('\n')
order_match = all(a == b for a, b in zip(gene_names_hd, gene_names_train))
print(f'\n[1] 유전자 순서 일치: {order_match}')
print(f'    첫 3개 hd:    {list(gene_names_hd[:3])}')
print(f'    첫 3개 train: {gene_names_train[:3]}')

# ── 2. tile_exprs 스케일 확인
val_exprs   = np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_01/tile_exprs.npy')
train_exprs = np.load('/project_antwerp/hbae/Loki_output/0228_epoch10_finetune_embedding/fold_01/train_exprs.npy')
print(f'\n[2] 발현값 스케일 비교:')
print(f'    tile_exprs:   mean={val_exprs.mean():.4f}, std={val_exprs.std():.4f}, max={val_exprs.max():.4f}')
print(f'    train_exprs:  mean={train_exprs.mean():.4f}, std={train_exprs.std():.4f}, max={train_exprs.max():.4f}')

# ── 3. 임베딩 정규화 확인
val_embs   = np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_01/tile_img_embs.npy')
train_embs = np.load('/project_antwerp/hbae/Loki_output/0228_epoch10_finetune_embedding/fold_01/train_text_embs.npy')
print(f'\n[3] 임베딩 norm 확인 (1이어야 함):')
print(f'    val_embs norm (첫 3개):   {np.linalg.norm(val_embs[:3], axis=1)}')
print(f'    train_embs norm (첫 3개): {np.linalg.norm(train_embs[:3], axis=1)}')

# ── 4. 유사도 분포 확인
val_t   = torch.tensor(val_embs).float()
train_t = torch.tensor(train_embs).float()
val_n   = F.normalize(val_t, dim=-1)
train_n = F.normalize(train_t, dim=-1)
sim = val_n[:5] @ train_n.T
print(f'\n[4] 유사도 분포 (첫 5개 타일):')
for i in range(5):
    print(f'    타일 {i}: min={sim[i].min():.3f}, max={sim[i].max():.3f}, mean={sim[i].mean():.3f}')

# ── 5. bin aggregation 좌표 확인
scalef = 0.09202595
df = pd.read_parquet('/project_antwerp/hbae/data/visium_hd_tonsil/binned_outputs/square_008um/spatial/tissue_positions.parquet', engine='pyarrow')
df = df[df['in_tissue']==1]
df['hires_col'] = df['pxl_col_in_fullres'] * scalef
df['hires_row'] = df['pxl_row_in_fullres'] * scalef
coords = np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_01/tile_coords.npy')
tile_size = 57
bin_col = df['hires_col'].values
bin_row = df['hires_row'].values
in_tile = (
    (bin_col >= coords[0,0]) & (bin_col < coords[0,0]+tile_size) &
    (bin_row >= coords[0,1]) & (bin_row < coords[0,1]+tile_size)
)
print(f'\n[5] 첫 번째 타일 bin 수: {in_tile.sum()}')
print(f'    타일 좌표: col={coords[0,0]}, row={coords[0,1]}')
print(f'    bin col 범위: {bin_col[in_tile].min():.1f} ~ {bin_col[in_tile].max():.1f}')
print(f'    bin row 범위: {bin_row[in_tile].min():.1f} ~ {bin_row[in_tile].max():.1f}')


파이프라인 검증

[1] 유전자 순서 일치: True
    첫 3개 hd:    ['A2M', 'A2ML1', 'AATK']
    첫 3개 train: ['A2M', 'A2ML1', 'AATK']

[2] 발현값 스케일 비교:
    tile_exprs:   mean=0.0323, std=0.1030, max=4.2285
    train_exprs:  mean=0.2906, std=0.6365, max=8.2888

[3] 임베딩 norm 확인 (1이어야 함):
    val_embs norm (첫 3개):   [1. 1. 1.]
    train_embs norm (첫 3개): [1. 1. 1.]

[4] 유사도 분포 (첫 5개 타일):
    타일 0: min=-0.112, max=0.425, mean=0.197
    타일 1: min=-0.124, max=0.426, mean=0.193
    타일 2: min=-0.092, max=0.436, mean=0.199
    타일 3: min=-0.087, max=0.441, mean=0.203
    타일 4: min=-0.079, max=0.430, mean=0.205

[5] 첫 번째 타일 bin 수: 378
    타일 좌표: col=2233, row=659
    bin col 범위: 2233.7 ~ 2287.6
    bin row 범위: 669.0 ~ 714.9


In [2]:

import numpy as np
import scanpy as sc

adata = sc.read_10x_h5('/project_antwerp/hbae/data/visium_hd_tonsil/binned_outputs/square_008um/filtered_feature_bc_matrix.h5')
adata.var_names_make_unique()

# normalize 전 count 분포
X_raw = adata.X
print('normalize 전 bin당 total count:')
counts = np.array(X_raw.sum(axis=1)).flatten()
print(f'  mean: {counts.mean():.1f}')
print(f'  median: {np.median(counts):.1f}')
print(f'  min: {counts.min():.1f}')
print(f'  max: {counts.max():.1f}')


/opt/conda/lib/python3.11/site-packages/anndata/_core/anndata.py:1798: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


normalize 전 bin당 total count:
  mean: 545.1
  median: 522.0
  min: 0.0
  max: 4339.0


/opt/conda/lib/python3.11/site-packages/anndata/_core/anndata.py:1798: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


In [1]:

import numpy as np

gene_names = open('/project_antwerp/hbae/data/0228_HVG_NEW/0228_HVG_Finetune_gene_list_full.txt').read().strip().split('\n')

# HNSCC train top 20 genes
train_exprs = np.load('/project_antwerp/hbae/Loki_output/0228_epoch10_finetune_embedding/fold_01/train_exprs.npy')
train_mean  = train_exprs.mean(axis=0)
train_top20 = np.argsort(train_mean)[::-1][:20]

# Visium HD top 20 genes
hd_exprs = np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_01/tile_exprs.npy')
hd_mean  = hd_exprs.mean(axis=0)
hd_top20 = np.argsort(hd_mean)[::-1][:20]

print('HNSCC train top 20 genes:')
for i, idx in enumerate(train_top20):
    print(f'  {i+1:2d}. {gene_names[idx]:<15} mean={train_mean[idx]:.4f}')

print()
print('Visium HD top 20 genes:')
for i, idx in enumerate(hd_top20):
    print(f'  {i+1:2d}. {gene_names[idx]:<15} mean={hd_mean[idx]:.4f}')

print()
# 겹치는 유전자
train_top20_genes = set([gene_names[i] for i in train_top20])
hd_top20_genes    = set([gene_names[i] for i in hd_top20])
overlap = train_top20_genes & hd_top20_genes
print(f'겹치는 유전자 ({len(overlap)}개): {overlap}')


HNSCC train top 20 genes:
   1. S100A8          mean=3.9134
   2. S100A9          mean=3.7616
   3. IGKC            mean=3.4091
   4. KRT16           mean=3.0061
   5. KRT6B           mean=2.7969
   6. PERP            mean=2.4547
   7. CD74            mean=2.4458
   8. COL1A1          mean=2.1131
   9. COL1A2          mean=2.0799
  10. SPRR1B          mean=2.0638
  11. CSTA            mean=2.0564
  12. IGHA1           mean=2.0496
  13. IGHG1           mean=1.9461
  14. IGHG3           mean=1.9343
  15. COL3A1          mean=1.8659
  16. KRTDAP          mean=1.8639
  17. PI3             mean=1.8399
  18. LY6D            mean=1.8384
  19. VIM             mean=1.8108
  20. SPARC           mean=1.8071

Visium HD top 20 genes:
   1. IGKC            mean=1.9753
   2. CD74            mean=1.7078
   3. IGHG1           mean=0.9045
   4. CORO1A          mean=0.8858
   5. CLU             mean=0.7368
   6. VIM             mean=0.7185
   7. IGHA1           mean=0.6263
   8. LTB             mean=0.59

In [2]:

import numpy as np

gene_names = open('/project_antwerp/hbae/data/0228_HVG_NEW/0228_HVG_Finetune_gene_list_full.txt').read().strip().split('\n')

train_exprs = np.load('/project_antwerp/hbae/Loki_output/0228_epoch10_finetune_embedding/fold_01/train_exprs.npy')
hd_exprs    = np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_01/tile_exprs.npy')

train_mean = train_exprs.mean(axis=0)
hd_mean    = hd_exprs.mean(axis=0)

# 상관관계 확인
from scipy.stats import pearsonr, spearmanr
r_pearson,  _ = pearsonr(train_mean, hd_mean)
r_spearman, _ = spearmanr(train_mean, hd_mean)
print(f'전체 2000개 유전자 평균 발현값 상관관계:')
print(f'  Pearson:  {r_pearson:.4f}')
print(f'  Spearman: {r_spearman:.4f}')

# top 100, 200, 500, 1000 겹치는 유전자 수
for k in [50, 100, 200, 500, 1000]:
    train_topk = set([gene_names[i] for i in np.argsort(train_mean)[::-1][:k]])
    hd_topk    = set([gene_names[i] for i in np.argsort(hd_mean)[::-1][:k]])
    overlap    = train_topk & hd_topk
    print(f'  top {k:4d} 겹치는 유전자: {len(overlap):3d}개 ({len(overlap)/k*100:.1f}%)')

# 발현값 분포 비교
print()
print('발현값 > 0.5인 유전자 수:')
print(f'  HNSCC train: {(train_mean > 0.5).sum()}개')
print(f'  Visium HD:   {(hd_mean > 0.5).sum()}개')

print()
print('HNSCC에서 높은데 Visium HD에서 낮은 유전자 (top 10):')
diff = train_mean - hd_mean
top_diff = np.argsort(diff)[::-1][:10]
for idx in top_diff:
    print(f'  {gene_names[idx]:<15} train={train_mean[idx]:.3f}, hd={hd_mean[idx]:.3f}')

print()
print('Visium HD에서 높은데 HNSCC에서 낮은 유전자 (top 10):')
top_diff2 = np.argsort(diff)[:10]
for idx in top_diff2:
    print(f'  {gene_names[idx]:<15} train={train_mean[idx]:.3f}, hd={hd_mean[idx]:.3f}')


전체 2000개 유전자 평균 발현값 상관관계:
  Pearson:  0.4003
  Spearman: 0.4011
  top   50 겹치는 유전자:  15개 (30.0%)
  top  100 겹치는 유전자:  24개 (24.0%)
  top  200 겹치는 유전자:  63개 (31.5%)
  top  500 겹치는 유전자: 211개 (42.2%)
  top 1000 겹치는 유전자: 653개 (65.3%)

발현값 > 0.5인 유전자 수:
  HNSCC train: 332개
  Visium HD:   8개

HNSCC에서 높은데 Visium HD에서 낮은 유전자 (top 10):
  S100A8          train=3.913, hd=0.308
  S100A9          train=3.762, hd=0.380
  KRT16           train=3.006, hd=0.046
  KRT6B           train=2.797, hd=0.020
  PERP            train=2.455, hd=0.070
  SPRR1B          train=2.064, hd=0.022
  CSTA            train=2.056, hd=0.085
  COL1A2          train=2.080, hd=0.202
  KRTDAP          train=1.864, hd=0.002
  COL1A1          train=2.113, hd=0.285

Visium HD에서 높은데 HNSCC에서 낮은 유전자 (top 10):
  CORO1A          train=0.429, hd=0.886
  LTB             train=0.220, hd=0.590
  CLU             train=0.490, hd=0.737
  CD37            train=0.178, hd=0.419
  CCL19           train=0.242, hd=0.476
  RASGRP2         train=0.094,

In [3]:

import numpy as np
import pandas as pd

scalef = 0.09202595
df = pd.read_parquet('/project_antwerp/hbae/data/visium_hd_tonsil/binned_outputs/square_008um/spatial/tissue_positions.parquet', engine='pyarrow')
df = df[df['in_tissue']==1].reset_index(drop=True)
df['hires_col'] = df['pxl_col_in_fullres'] * scalef
df['hires_row'] = df['pxl_row_in_fullres'] * scalef

bin_col = df['hires_col'].values
bin_row = df['hires_row'].values

coords    = np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_01/tile_coords.npy')
tile_size = 57

# 각 타일의 bin 수 + 이미지 밝기 함께 확인
from PIL import Image
img = np.array(Image.open('/project_antwerp/hbae/data/visium_hd_tonsil/spatial/tissue_hires_image.png').convert('RGB'))

bin_counts   = []
brightnesses = []

for col_start, row_start in coords:
    in_tile = (
        (bin_col >= col_start) & (bin_col < col_start + tile_size) &
        (bin_row >= row_start) & (bin_row < row_start + tile_size)
    )
    bin_counts.append(in_tile.sum())
    patch = img[int(row_start):int(row_start)+tile_size, int(col_start):int(col_start)+tile_size]
    brightnesses.append(patch.mean())

bin_counts   = np.array(bin_counts)
brightnesses = np.array(brightnesses)

print('bin 수 분포:')
print(f'  min: {bin_counts.min()}, max: {bin_counts.max()}, mean: {bin_counts.mean():.1f}')
print(f'  bin < 50:  {(bin_counts < 50).sum()}개 타일')
print(f'  bin < 100: {(bin_counts < 100).sum()}개 타일')
print(f'  bin >= 300: {(bin_counts >= 300).sum()}개 타일')

print()
print('이미지 밝기 분포:')
print(f'  min: {brightnesses.min():.1f}, max: {brightnesses.max():.1f}, mean: {brightnesses.mean():.1f}')
print(f'  밝기 > 200 (배경 의심): {(brightnesses > 200).sum()}개 타일')
print(f'  밝기 > 220 (거의 배경): {(brightnesses > 220).sum()}개 타일')

# bin 수 적은데 밝기 높은 타일 (문제 타일)
problem = (bin_counts < 100) | (brightnesses > 200)
print(f'\n문제 타일 (bin<100 OR 밝기>200): {problem.sum()}개')


bin 수 분포:
  min: 121, max: 484, mean: 438.5
  bin < 50:  0개 타일
  bin < 100: 0개 타일
  bin >= 300: 1528개 타일

이미지 밝기 분포:
  min: 102.8, max: 220.6, mean: 129.3
  밝기 > 200 (배경 의심): 4개 타일
  밝기 > 220 (거의 배경): 1개 타일

문제 타일 (bin<100 OR 밝기>200): 4개


In [4]:

import pandas as pd

df = pd.read_parquet('/project_antwerp/hbae/data/visium_hd_tonsil/binned_outputs/square_008um/spatial/tissue_positions.parquet', engine='pyarrow')

print('전체 bin 수:', len(df))
print('in_tissue=1:', df.in_tissue.sum())
print('in_tissue=0:', (df.in_tissue==0).sum())
print('in_tissue 비율:', df.in_tissue.mean())
print()
print('샘플:')
print(df.head(3))


전체 bin 수: 702244
in_tissue=1: 701631
in_tissue=0: 613
in_tissue 비율: 0.9991270840334698

샘플:
                 barcode  in_tissue  array_row  array_col  pxl_row_in_fullres  \
0  s_008um_00000_00000-1          1          0          0        31731.475716   
1  s_008um_00000_00001-1          1          0          1        31731.353265   
2  s_008um_00000_00002-1          1          0          2        31731.230813   

   pxl_col_in_fullres  
0        24376.911561  
1        24406.138440  
2        24435.365319  


In [5]:

import numpy as np
import torch
import torch.nn.functional as F
from scipy.stats import pearsonr
import scanpy as sc

# Visium HD 전체 유전자로 count matrix 로드
adata = sc.read_10x_h5('/project_antwerp/hbae/data/visium_hd_tonsil/binned_outputs/square_008um/filtered_feature_bc_matrix.h5')
adata.var_names_make_unique()
print('Visium HD 전체 유전자 수:', adata.shape[1])

# Visium HD HVG 선택
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=2000)
hd_hvg = adata.var_names[adata.var['highly_variable']].tolist()

# HNSCC HVG와 비교
hnscc_hvg = open('/project_antwerp/hbae/data/0228_HVG_NEW/0228_HVG_Finetune_gene_list_full.txt').read().strip().split('\n')

overlap = set(hd_hvg) & set(hnscc_hvg)
print(f'Visium HD HVG 2000개')
print(f'HNSCC HVG 2000개')
print(f'겹치는 유전자: {len(overlap)}개 ({len(overlap)/2000*100:.1f}%)')
print(f'Visium HD HVG top 20: {hd_hvg[:20]}')


/opt/conda/lib/python3.11/site-packages/anndata/_core/anndata.py:1798: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/opt/conda/lib/python3.11/site-packages/anndata/_core/anndata.py:1798: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


Visium HD 전체 유전자 수: 18085


/opt/conda/lib/python3.11/site-packages/legacy_api_wrap/__init__.py:88: UserWarning: Some cells have zero counts
  return fn(*args_all, **kw)


Visium HD HVG 2000개
HNSCC HVG 2000개
겹치는 유전자: 201개 (10.1%)
Visium HD HVG top 20: ['PLEKHN1', 'HES4', 'RNF223', 'TMEM52', 'CCDC27', 'CHD5', 'SLC2A7', 'DRAXIN', 'C1orf167', 'NPPA', 'C1orf158', 'PRAMEF27', 'HNRNPCL2', 'PRAMEF8', 'PRAMEF15', 'PRAMEF14', 'KAZN', 'SPATA21', 'PADI1', 'PADI3']


In [7]:

import numpy as np
import torch
import torch.nn.functional as F
from scipy.stats import pearsonr

train_embs  = torch.tensor(np.load('/project_antwerp/hbae/Loki_output/0228_epoch10_finetune_embedding/fold_01/train_text_embs.npy')).float()
train_exprs = torch.tensor(np.load('/project_antwerp/hbae/Loki_output/0228_epoch10_finetune_embedding/fold_01/train_exprs.npy')).float()
val_embs    = torch.tensor(np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_01/tile_img_embs.npy')).float()
val_exprs   = np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_01/tile_exprs.npy')

train_norm = F.normalize(train_embs, dim=-1)
val_norm   = F.normalize(val_embs, dim=-1)

gene_names = open('/project_antwerp/hbae/data/0228_HVG_NEW/0228_HVG_Finetune_gene_list_full.txt').read().strip().split('\n')

# 예측
preds = []
for i in range(len(val_norm)):
    sim = val_norm[i] @ train_norm.T
    idx = sim.topk(500).indices
    w   = sim[idx] / sim[idx].sum()
    preds.append((w[:, None] * train_exprs[idx]).sum(0).numpy())
preds = np.array(preds)

# 발현값 > 0.5인 유전자만
mean_expr = val_exprs.mean(axis=0)
expressed_idx = np.where(mean_expr > 0.5)[0]
print(f'발현값 > 0.5 유전자 수: {len(expressed_idx)}개')
print()

gene_pccs = []
for idx in expressed_idx:
    if val_exprs[:, idx].std() > 1e-8:
        r, _ = pearsonr(preds[:, idx], val_exprs[:, idx])
        if np.isfinite(r):
            gene_pccs.append((gene_names[idx], r, mean_expr[idx]))

gene_pccs.sort(key=lambda x: -x[1])
print('발현되는 유전자들의 Gene-wise PCC:')
print('gene            PCC     mean_expr')
print('-'*45)
for g, r, m in gene_pccs:
    print(g.ljust(15), round(r,4), round(m,4))
print()
print(f'평균 PCC: {np.mean([x[1] for x in gene_pccs]):.4f}')


발현값 > 0.5 유전자 수: 8개

발현되는 유전자들의 Gene-wise PCC:
gene            PCC     mean_expr
---------------------------------------------
CORO1A          0.0347 0.8858
VIM             0.0302 0.7185
CLU             0.0152 0.7368
LTB             0.0055 0.5903
IGKC            -0.0063 1.9753
CD74            -0.0322 1.7078
IGHG1           -0.0414 0.9045
IGHA1           -0.0808 0.6263

평균 PCC: -0.0094


In [8]:

import numpy as np
import torch
import torch.nn.functional as F
from scipy.stats import pearsonr

train_embs  = torch.tensor(np.load('/project_antwerp/hbae/Loki_output/0228_epoch10_finetune_embedding/fold_01/train_text_embs.npy')).float()
train_exprs = torch.tensor(np.load('/project_antwerp/hbae/Loki_output/0228_epoch10_finetune_embedding/fold_01/train_exprs.npy')).float()
val_embs    = torch.tensor(np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_01/tile_img_embs.npy')).float()
val_exprs   = np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_01/tile_exprs.npy')

train_norm = F.normalize(train_embs, dim=-1)
val_norm   = F.normalize(val_embs, dim=-1)
gene_names = open('/project_antwerp/hbae/data/0228_HVG_NEW/0228_HVG_Finetune_gene_list_full.txt').read().strip().split('\n')

preds = []
for i in range(len(val_norm)):
    sim = val_norm[i] @ train_norm.T
    idx = sim.topk(500).indices
    w   = sim[idx] / sim[idx].sum()
    preds.append((w[:, None] * train_exprs[idx]).sum(0).numpy())
preds = np.array(preds)

# 전체 2000개 유전자 gene-wise PCC
mean_expr  = val_exprs.mean(axis=0)
top300_idx = np.argsort(mean_expr)[::-1][:300]

gene_pccs = []
for idx in top300_idx:
    if val_exprs[:, idx].std() > 1e-8:
        r, _ = pearsonr(preds[:, idx], val_exprs[:, idx])
        if np.isfinite(r):
            gene_pccs.append((gene_names[idx], r, mean_expr[idx]))

gene_pccs.sort(key=lambda x: -x[1])

print('Gene-wise PCC 상위 20개:')
print('gene            PCC     mean_expr')
print('-'*45)
for g, r, m in gene_pccs[:20]:
    print(g.ljust(15), round(r,4), round(m,4))

print()
print(f'PCC > 0.1인 유전자: {sum(1 for _,r,_ in gene_pccs if r > 0.1)}개')
print(f'PCC > 0.2인 유전자: {sum(1 for _,r,_ in gene_pccs if r > 0.2)}개')
print(f'PCC > 0.0인 유전자: {sum(1 for _,r,_ in gene_pccs if r > 0.0)}개')


Gene-wise PCC 상위 20개:
gene            PCC     mean_expr
---------------------------------------------
PTGDS           0.1774 5.325
C3              0.167 5.0425
CXCL14          0.1653 4.3962
ZAP70           0.1517 4.1573
TXNIP           0.1454 5.0562
CD8A            0.1446 3.6746
ABI3BP          0.1406 3.5459
TSPAN4          0.1405 3.4274
CD6             0.1387 4.206
TRAF3IP3        0.1354 4.3418
CYFIP2          0.1323 3.5018
TRAC            0.132 5.0225
GIMAP7          0.131 3.71
ADRA2A          0.1307 4.0652
TPM2            0.1298 3.9477
GIMAP5          0.1274 3.5272
PSTPIP1         0.1261 3.8046
CELF2           0.1233 4.3207
TCF7            0.1218 4.3356
TRBC1           0.1213 4.5249

PCC > 0.1인 유전자: 44개
PCC > 0.2인 유전자: 0개
PCC > 0.0인 유전자: 227개


In [9]:

import numpy as np

gene_names  = open('/project_antwerp/hbae/data/0228_HVG_NEW/0228_HVG_Finetune_gene_list_full.txt').read().strip().split('\n')
train_exprs = np.load('/project_antwerp/hbae/Loki_output/0228_epoch10_finetune_embedding/fold_01/train_exprs.npy')
train_mean  = train_exprs.mean(axis=0)

# B세포 마커들 확인
b_cell_genes = ['IGKC', 'IGHG1', 'IGHA1', 'IGHG3', 'CD74', 'CD19', 'CD20', 'MS4A1',
                'IGLC1', 'IGLC2', 'IGHM', 'IGHA2', 'CD79A', 'CD79B', 'PAX5', 'CORO1A', 'LTB', 'CD37']

print('HNSCC train에서 B세포 유전자 발현:')
print('gene            mean_expr  rank')
print('-'*45)
rank_order = np.argsort(train_mean)[::-1]
rank_dict  = {gene_names[i]: r+1 for r, i in enumerate(rank_order)}

for g in b_cell_genes:
    if g in gene_names:
        idx  = gene_names.index(g)
        mean = train_mean[idx]
        rank = rank_dict[g]
        print(f'{g.ljust(15)} {mean:.4f}     {rank}위')
    else:
        print(f'{g.ljust(15)} HVG에 없음')


HNSCC train에서 B세포 유전자 발현:
gene            mean_expr  rank
---------------------------------------------
IGKC            3.4091     3위
IGHG1           1.9461     13위
IGHA1           2.0496     12위
IGHG3           1.9343     14위
CD74            2.4458     7위
CD19            HVG에 없음
CD20            HVG에 없음
MS4A1           HVG에 없음
IGLC1           1.5470     35위
IGLC2           HVG에 없음
IGHM            0.5226     316위
IGHA2           0.2232     701위
CD79A           0.1527     920위
CD79B           0.0798     1441위
PAX5            HVG에 없음
CORO1A          0.4290     383위
LTB             0.2199     707위
CD37            0.1775     833위


In [10]:

import numpy as np
import torch
import torch.nn.functional as F
from scipy.stats import pearsonr

train_exprs = np.load('/project_antwerp/hbae/Loki_output/0228_epoch10_finetune_embedding/fold_01/train_exprs.npy')
val_exprs   = np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_01/tile_exprs.npy')
gene_names  = open('/project_antwerp/hbae/data/0228_HVG_NEW/0228_HVG_Finetune_gene_list_full.txt').read().strip().split('\n')

igkc_idx = gene_names.index('IGKC')
cd74_idx = gene_names.index('CD74')
ptgds_idx = gene_names.index('PTGDS')
zap70_idx = gene_names.index('ZAP70')

print('HNSCC train spatial variation (spot간 std):')
print(f'  IGKC:  {train_exprs[:, igkc_idx].std():.4f}')
print(f'  CD74:  {train_exprs[:, cd74_idx].std():.4f}')
print(f'  PTGDS: {train_exprs[:, ptgds_idx].std():.4f}')
print(f'  ZAP70: {train_exprs[:, zap70_idx].std():.4f}')

print()
print('Visium HD spatial variation (tile간 std):')
print(f'  IGKC:  {val_exprs[:, igkc_idx].std():.4f}')
print(f'  CD74:  {val_exprs[:, cd74_idx].std():.4f}')
print(f'  PTGDS: {val_exprs[:, ptgds_idx].std():.4f}')
print(f'  ZAP70: {val_exprs[:, zap70_idx].std():.4f}')

print()
print('HNSCC train에서 IGKC 분포:')
igkc_train = train_exprs[:, igkc_idx]
print(f'  zeros 비율: {(igkc_train==0).mean():.4f}')
print(f'  > 1.0 비율: {(igkc_train > 1.0).mean():.4f}')
print(f'  > 3.0 비율: {(igkc_train > 3.0).mean():.4f}')

print()
print('Visium HD에서 IGKC 분포:')
igkc_val = val_exprs[:, igkc_idx]
print(f'  zeros 비율: {(igkc_val==0).mean():.4f}')
print(f'  > 1.0 비율: {(igkc_val > 1.0).mean():.4f}')
print(f'  > 3.0 비율: {(igkc_val > 3.0).mean():.4f}')


HNSCC train spatial variation (spot간 std):
  IGKC:  2.1417
  CD74:  1.1763
  PTGDS: 0.6176
  ZAP70: 0.3492

Visium HD spatial variation (tile간 std):
  IGKC:  1.1181
  CD74:  0.5176
  PTGDS: 1.3006
  ZAP70: 1.0772

HNSCC train에서 IGKC 분포:
  zeros 비율: 0.0824
  > 1.0 비율: 0.8338
  > 3.0 비율: 0.5492

Visium HD에서 IGKC 분포:
  zeros 비율: 0.0000
  > 1.0 비율: 1.0000
  > 3.0 비율: 1.0000


In [1]:

import json
import h5py

# scalefactors 확인
with open('/project_antwerp/hbae/data/visium_hd_tonsil/binned_outputs/square_008um/spatial/scalefactors_json.json') as f:
    sf = json.load(f)
print('scalefactors:')
for k, v in sf.items():
    print(f'  {k}: {v}')

# h5 metadata 확인
with h5py.File('/project_antwerp/hbae/data/visium_hd_tonsil/Visium_HD_FF_Human_Tonsil_feature_slice.h5', 'r') as f:
    meta = json.loads(f.attrs['metadata_json'])
print()
print('metadata:')
for k, v in meta.items():
    print(f'  {k}: {v}')


scalefactors:
  spot_diameter_fullres: 29.225301656814043
  bin_size_um: 8.0
  microns_per_pixel: 0.273735412347224
  tissue_lowres_scalef: 0.009202595
  fiducial_diameter_fullres: 1205.5436933435794
  tissue_hires_scalef: 0.09202595
  regist_target_img_scalef: 0.09202595

metadata:
  sample_id: Visium_HD_FF_Human_Tonsil
  sample_desc: Gene expression library of Fresh Frozen Human Tonsil (Visium HD) using the Human Whole Transcriptome Probe Set
  slide_name: visium_hd_v1
  nrows: 3350
  ncols: 3350
  spot_pitch: 2.0
  hd_layout_json: {"slide_uid":"H1-BHM9GT7","file_format":"1.0","aligner_version":"1.0","input_hash":"96bde14d95ba0743d7e9562b89160cac","slide_design":"visium_hd_rc1","transform":[1.0000325399274357,0.002591775464519526,-33.657701753274374,-0.0025907969752494314,1.0000102123948542,63.96893850375202,9.414438094981444e-9,-2.285391216086935e-9,1.0]}
  transform_matrices: {'spot_colrow_to_microscope_colrow': [[-3.65394524181981, 0.002133389590472481, 15524.64179357808], [0.0030

In [2]:

import numpy as np
import pandas as pd

scalef = 0.09202595
df = pd.read_parquet('/project_antwerp/hbae/data/visium_hd_tonsil/binned_outputs/square_008um/spatial/tissue_positions.parquet', engine='pyarrow')
df = df[df['in_tissue']==1]
df['hires_col'] = df['pxl_col_in_fullres'] * scalef
df['hires_row'] = df['pxl_row_in_fullres'] * scalef
bin_col = df['hires_col'].values
bin_row = df['hires_row'].values

coords_23 = np.load('/project_antwerp/hbae/Loki_output/visium_hd_23px_embeddings/fold_01/tile_coords.npy')
coords_57 = np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_01/tile_coords.npy')

# 23px 타일당 bin 수
counts_23 = []
for col_start, row_start in coords_23[:100]:
    in_tile = (
        (bin_col >= col_start) & (bin_col < col_start + 23) &
        (bin_row >= row_start) & (bin_row < row_start + 23)
    )
    counts_23.append(in_tile.sum())

counts_57 = []
for col_start, row_start in coords_57[:100]:
    in_tile = (
        (bin_col >= col_start) & (bin_col < col_start + 57) &
        (bin_row >= row_start) & (bin_row < row_start + 57)
    )
    counts_57.append(in_tile.sum())

print(f'23px 타일당 bin 수: mean={np.mean(counts_23):.1f}, min={np.min(counts_23)}')
print(f'57px 타일당 bin 수: mean={np.mean(counts_57):.1f}, min={np.min(counts_57)}')

# 8um bin이 23px 안에 몇 개 들어가는지
um_per_px_hires = 0.2737 / 0.0920
print(f'\n23px = {23 * um_per_px_hires:.1f}um')
print(f'57px = {57 * um_per_px_hires:.1f}um')
print(f'8um bin 1개 = {8 / um_per_px_hires:.1f}px')
print(f'23px 안에 들어가는 bin 수 (이론): {(23 / (8/um_per_px_hires))**2:.1f}개')
print(f'57px 안에 들어가는 bin 수 (이론): {(57 / (8/um_per_px_hires))**2:.1f}개')


23px 타일당 bin 수: mean=60.3, min=40
57px 타일당 bin 수: mean=432.1, min=231

23px = 68.4um
57px = 169.6um
8um bin 1개 = 2.7px
23px 안에 들어가는 bin 수 (이론): 73.2개
57px 안에 들어가는 bin 수 (이론): 449.3개


In [1]:

import pandas as pd
import numpy as np

# 2um bin 확인
df2 = pd.read_parquet('/project_antwerp/hbae/data/visium_hd_tonsil/binned_outputs/square_002um/spatial/tissue_positions.parquet', engine='pyarrow')
df2 = df2[df2['in_tissue']==1]
df2['hires_col'] = df2['pxl_col_in_fullres'] * 0.09202595
df2['hires_row'] = df2['pxl_row_in_fullres'] * 0.09202595

# 16um bin 확인
df16 = pd.read_parquet('/project_antwerp/hbae/data/visium_hd_tonsil/binned_outputs/square_016um/spatial/tissue_positions.parquet', engine='pyarrow')
df16 = df16[df16['in_tissue']==1]
df16['hires_col'] = df16['pxl_col_in_fullres'] * 0.09202595
df16['hires_row'] = df16['pxl_row_in_fullres'] * 0.09202595

um_per_px = 0.2737 / 0.0920
tile_size = 23

print('=== 2um bin ===')
print(f'총 bin 수: {len(df2):,}')
bin_col = df2.hires_col.values
bin_row = df2.hires_row.values
# 첫 타일 bin 수
col_start = int(bin_col.min())
row_start = int(bin_row.min())
in_tile = (
    (bin_col >= col_start) & (bin_col < col_start + tile_size) &
    (bin_row >= row_start) & (bin_row < row_start + tile_size)
)
print(f'첫 타일 bin 수: {in_tile.sum()}')
print(f'이론 bin 수: {(tile_size / (2/um_per_px))**2:.0f}개')

print()
print('=== 16um bin ===')
print(f'총 bin 수: {len(df16):,}')
bin_col = df16.hires_col.values
bin_row = df16.hires_row.values
in_tile = (
    (bin_col >= col_start) & (bin_col < col_start + tile_size) &
    (bin_row >= row_start) & (bin_row < row_start + tile_size)
)
print(f'첫 타일 bin 수: {in_tile.sum()}')
print(f'이론 bin 수: {(tile_size / (16/um_per_px))**2:.0f}개')

print()
print(f'참고: 2um bin 1개 = {2/um_per_px:.2f}px')
print(f'참고: 16um bin 1개 = {16/um_per_px:.2f}px')
print(f'참고: 23px 타일 = {23 * um_per_px:.1f}um')


=== 2um bin ===
총 bin 수: 11,210,791
첫 타일 bin 수: 714
이론 bin 수: 1170개

=== 16um bin ===
총 bin 수: 175,448
첫 타일 bin 수: 12
이론 bin 수: 18개

참고: 2um bin 1개 = 0.67px
참고: 16um bin 1개 = 5.38px
참고: 23px 타일 = 68.4um


In [2]:

import h5py
import os

f2  = '/project_antwerp/hbae/data/visium_hd_tonsil/binned_outputs/square_002um/filtered_feature_bc_matrix.h5'
f8  = '/project_antwerp/hbae/data/visium_hd_tonsil/binned_outputs/square_008um/filtered_feature_bc_matrix.h5'

print('파일 크기:')
print(f'  2um h5: {os.path.getsize(f2)/1e9:.2f} GB')
print(f'  8um h5: {os.path.getsize(f8)/1e9:.2f} GB')

with h5py.File(f2, 'r') as f:
    shape = f['matrix/data'].shape
    print(f'  2um nnz: {shape[0]:,}')


파일 크기:
  2um h5: 0.64 GB
  8um h5: 0.42 GB
  2um nnz: 350,677,940


In [3]:

import numpy as np

exprs_57 = np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_01/tile_exprs.npy')
exprs_23_8um = np.load('/project_antwerp/hbae/Loki_output/visium_hd_23px_embeddings/fold_01/tile_exprs.npy')
exprs_23_2um = np.load('/project_antwerp/hbae/Loki_output/visium_hd_23px_2um_embeddings/fold_01/tile_exprs.npy')

print('57px + 8um bin:')
print(f'  mean: {exprs_57.mean():.4f}, std: {exprs_57.std():.4f}')
print(f'  zeros: {(exprs_57==0).mean():.4f}')

print()
print('23px + 8um bin:')
print(f'  mean: {exprs_23_8um.mean():.4f}, std: {exprs_23_8um.std():.4f}')
print(f'  zeros: {(exprs_23_8um==0).mean():.4f}')

print()
print('23px + 2um bin:')
print(f'  mean: {exprs_23_2um.mean():.4f}, std: {exprs_23_2um.std():.4f}')
print(f'  zeros: {(exprs_23_2um==0).mean():.4f}')


57px + 8um bin:
  mean: 1.8828, std: 1.5872
  zeros: 0.2458

23px + 8um bin:
  mean: 0.7960, std: 1.0648
  zeros: 0.5113

23px + 2um bin:
  mean: 0.7976, std: 1.0664
  zeros: 0.5111


In [5]:

from open_clip import create_model_from_pretrained
import numpy as np
from PIL import Image

model, preprocess = create_model_from_pretrained(
    'coca_ViT-L-14',
    pretrained='/project_antwerp/assets/loki_ckpts/checkpoint.pt',weights_only=False
)

# preprocess 내용 확인
print(preprocess)

# 실제로 어떻게 변환되는지 확인
patch_57 = np.zeros((57, 57, 3), dtype=np.uint8)
patch_23 = np.zeros((23, 23, 3), dtype=np.uint8)

out_57 = preprocess(Image.fromarray(patch_57))
out_23 = preprocess(Image.fromarray(patch_23))
print(f'57px 입력 → 출력 shape: {out_57.shape}')
print(f'23px 입력 → 출력 shape: {out_23.shape}')


Compose(
    Resize(size=224, interpolation=bicubic, max_size=None, antialias=True)
    CenterCrop(size=(224, 224))
    <function _convert_to_rgb at 0x7f71173063e0>
    ToTensor()
    Normalize(mean=(0.48145466, 0.4578275, 0.40821073), std=(0.26862954, 0.26130258, 0.27577711))
)
57px 입력 → 출력 shape: torch.Size([3, 224, 224])
23px 입력 → 출력 shape: torch.Size([3, 224, 224])


In [6]:

import pandas as pd
import os

# GSE281978 샘플 목록 확인
base = '/project_antwerp/hbae/data'
for d in os.listdir(base):
    if 'GSE281978' in d:
        print(d)

# fold_01 train 샘플 확인
fold_csv = '/project_antwerp/hbae/Loki_output/10fold_csv_file/0317_New_HVG_10fold/fold_01_train.csv'
if os.path.exists(fold_csv):
    df = pd.read_csv(fold_csv)
    print(df.head())
    gse281978 = df[df['sample_id'].str.contains('GSE281978', na=False)]
    print(f'GSE281978 샘플 수: {len(gse281978)}')
    print(gse281978['sample_id'].unique())


                                            img_path  \
0  /project_antwerp/hbae/data/Processed_Data/GSE1...   
1  /project_antwerp/hbae/data/Processed_Data/GSE1...   
2  /project_antwerp/hbae/data/Processed_Data/GSE1...   
3  /project_antwerp/hbae/data/Processed_Data/GSE1...   
4  /project_antwerp/hbae/data/Processed_Data/GSE1...   

                                               label  
0  MYH7 MYH2 MYBPC1 KLHL41 ACTN2 SLN HSPB6 NEB CK...  
1  WFDC1 ELN TNNC2 SHC3 PLA2G2A ACTN2 ABI3BP PILR...  
2  AOC3 CCL14 ACKR1 ABCC9 GATA6 ARHGEF15 ELN MYH7...  
3  TNNC2 CHRNA1 ABCA8 MYBPC1 MYBPC2 CKM EEF1A2 PR...  
4  IGF1 CCN5 ACKR1 FGF7 SVEP1 GPR4 PRELP AOC3 ABC...  


KeyError: 'sample_id'

In [11]:

import numpy as np
import scanpy as sc
from scipy.stats import pearsonr, spearmanr

gene_names = open('/project_antwerp/hbae/data/0228_HVG_NEW/0228_HVG_Finetune_gene_list_full.txt').read().strip().split('\n')
hd_exprs   = np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_01/tile_exprs.npy')
hd_mean    = hd_exprs.mean(axis=0)

samples = {
    'GSM8633891_PT_HPV-':   'GSM8633891_21_00757_LI_SING',
    'GSM8633892_LNMT_HPV-': 'GSM8633892_21_00758_LI_SING',
    'GSM8633893_PT_HPV+':   'GSM8633893_21_01569_LI_SING',
    'GSM8633894_LNMT_HPV+': 'GSM8633894_21_01570_LI_SING',
    'GSM8633895_PT_HPV-':   'GSM8633895_21_01586_LI_SING',
    'GSM8633896_LNMT_HPV-': 'GSM8633896_21_01587_LI_SING',
}

base = '/project_antwerp/hbae/data/Processed_Data/GSE281978'

print('샘플                           Pearson  Spearman  top5 genes')
print('-'*85)

for label, sid in samples.items():
    path = f'{base}/{sid}/st_norm_noHK.h5ad'
    adata = sc.read_h5ad(path)

    common      = [g for g in gene_names if g in adata.var_names]
    adata       = adata[:, common]
    gene_to_idx = {g: i for i, g in enumerate(gene_names)}
    sample_mean = np.zeros(len(gene_names))
    for g in common:
        gi = gene_to_idx[g]
        sample_mean[gi] = float(np.array(adata[:, g].X.mean()))

    r_p, _ = pearsonr(sample_mean, hd_mean)
    r_s, _ = spearmanr(sample_mean, hd_mean)
    top5   = [gene_names[i] for i in np.argsort(sample_mean)[::-1][:5]]

    print(f'{label:<30} {r_p:>8.4f} {r_s:>8.4f}   {top5}')


샘플                           Pearson  Spearman  top5 genes
-------------------------------------------------------------------------------------
GSM8633891_PT_HPV-               0.2693   0.2941   ['IGKC', 'COL1A1', 'COL1A2', 'SPP1', 'COL3A1']
GSM8633892_LNMT_HPV-             0.2554   0.2675   ['COL1A1', 'COL3A1', 'COL1A2', 'FN1', 'SPARC']
GSM8633893_PT_HPV+               0.5894   0.7226   ['IGKC', 'CD74', 'IGHG1', 'IGHG3', 'KRT15']
GSM8633894_LNMT_HPV+             0.4476   0.6372   ['IGKC', 'KRT15', 'CD74', 'IGHG1', 'KRT13']
GSM8633895_PT_HPV-               0.2372   0.2436   ['NDRG1', 'TGFBI', 'LGALS1', 'LAMB3', 'SLC2A1']
GSM8633896_LNMT_HPV-             0.3125   0.3511   ['CSTA', 'IGKC', 'COL1A1', 'CD9', 'FN1']


In [5]:

import json
import pandas as pd
import numpy as np

with open('/project_antwerp/hbae/data/visium_hd_tonsil/binned_outputs/square_008um/spatial/scalefactors_json.json') as f:
    sf = json.load(f)

print('scalefactors:')
print(f"  microns_per_pixel:   {sf['microns_per_pixel']}")
print(f"  tissue_hires_scalef: {sf['tissue_hires_scalef']}")

# bin 좌표 확인 (fullres vs hires)
df = pd.read_parquet('/project_antwerp/hbae/data/visium_hd_tonsil/binned_outputs/square_008um/spatial/tissue_positions.parquet', engine='pyarrow')
df = df[df['in_tissue']==1].head(5)
print()
print('bin 좌표 샘플:')
print(f'  pxl_col_in_fullres: {df.pxl_col_in_fullres.values[:3]}')
hires_col = df.pxl_col_in_fullres.values[:3] * sf['tissue_hires_scalef']
print(f'  × scalef → hires:   {hires_col}')

# fullres 기준 타일 크기
um_per_px_fullres = sf['microns_per_pixel']
tile_fullres = 68.0 / um_per_px_fullres
tile_hires   = tile_fullres * sf['tissue_hires_scalef']
print()
print(f'fullres 기준: 68um = {tile_fullres:.1f}px')
print(f'hires 기준:   68um = {tile_hires:.1f}px')
print()
print('→ bin 좌표를 hires로 변환 후 23px 타일 사용 = 맞음')
print('→ bin 좌표를 fullres 그대로 쓰면 248px 타일 사용해야 함')


scalefactors:
  microns_per_pixel:   0.273735412347224
  tissue_hires_scalef: 0.09202595

bin 좌표 샘플:
  pxl_col_in_fullres: [24376.9115614  24406.1384402  24435.36531867]
  × scalef → hires:   [2243.3084445  2245.99807579 2248.68770705]

fullres 기준: 68um = 248.4px
hires 기준:   68um = 22.9px

→ bin 좌표를 hires로 변환 후 23px 타일 사용 = 맞음
→ bin 좌표를 fullres 그대로 쓰면 248px 타일 사용해야 함


In [6]:

import numpy as np
import pandas as pd
import json

with open('/project_antwerp/hbae/data/visium_hd_tonsil/binned_outputs/square_008um/spatial/scalefactors_json.json') as f:
    sf = json.load(f)
scalef = sf['tissue_hires_scalef']

df = pd.read_parquet('/project_antwerp/hbae/data/visium_hd_tonsil/binned_outputs/square_008um/spatial/tissue_positions.parquet', engine='pyarrow')
df = df[df['in_tissue']==1].reset_index(drop=True)

bin_col_full = df['pxl_col_in_fullres'].values
bin_row_full = df['pxl_row_in_fullres'].values
bin_col_hi   = bin_col_full * scalef
bin_row_hi   = bin_row_full * scalef

# 방법 1: hires 좌표, 23px 타일
col_start_hi = 2233.0
tile_hi      = 23
in_hires = (
    (bin_col_hi >= col_start_hi) & (bin_col_hi < col_start_hi + tile_hi) &
    (bin_row_hi >= 659.0)        & (bin_row_hi < 659.0 + tile_hi)
)

# 방법 2: fullres 좌표, 248px 타일
col_start_full = col_start_hi / scalef
tile_full      = tile_hi / scalef
in_full = (
    (bin_col_full >= col_start_full) & (bin_col_full < col_start_full + tile_full) &
    (bin_row_full >= 659.0/scalef)   & (bin_row_full < 659.0/scalef + tile_full)
)

print(f'방법 1 (hires, 23px):    bin 수 = {in_hires.sum()}')
print(f'방법 2 (fullres, 248px): bin 수 = {in_full.sum()}')
print(f'동일 여부: {(in_hires == in_full).all()}')


방법 1 (hires, 23px):    bin 수 = 45
방법 2 (fullres, 248px): bin 수 = 45
동일 여부: True


In [1]:

import numpy as np
import pandas as pd
import json

with open('/project_antwerp/hbae/data/visium_hd_tonsil/binned_outputs/square_008um/spatial/scalefactors_json.json') as f:
    sf = json.load(f)
scalef = sf['tissue_hires_scalef']

df = pd.read_parquet('/project_antwerp/hbae/data/visium_hd_tonsil/binned_outputs/square_008um/spatial/tissue_positions.parquet', engine='pyarrow')
df = df[df['in_tissue']==1].reset_index(drop=True)
df['hires_col'] = df['pxl_col_in_fullres'] * scalef
df['hires_row'] = df['pxl_row_in_fullres'] * scalef

bin_col = df['hires_col'].values
bin_row = df['hires_row'].values

# 23px 타일 첫 번째
col_start, row_start = 2233.0, 659.0
tile_size = 23

in_tile = (
    (bin_col >= col_start) & (bin_col < col_start + tile_size) &
    (bin_row >= row_start) & (bin_row < row_start + tile_size)
)

print(f'타일 범위: col {col_start:.1f}~{col_start+tile_size:.1f}, row {row_start:.1f}~{row_start+tile_size:.1f}')
print(f'실제 bin 수: {in_tile.sum()}')
print()

# bin 간격 확인
tile_bins_col = bin_col[in_tile]
tile_bins_row = bin_row[in_tile]
col_sorted = np.sort(np.unique(tile_bins_col))
row_sorted = np.sort(np.unique(tile_bins_row))
print(f'unique col 수: {len(col_sorted)}')
print(f'unique row 수: {len(row_sorted)}')
if len(col_sorted) > 1:
    print(f'col 간격: {np.diff(col_sorted[:5])}')
if len(row_sorted) > 1:
    print(f'row 간격: {np.diff(row_sorted[:5])}')

# 8um bin이 hires에서 몇 px인지
um_per_px_hires = sf['microns_per_pixel'] / sf['tissue_hires_scalef']
bin_px = 8.0 / um_per_px_hires
print(f'\n8um bin = {bin_px:.2f}px (hires)')
print(f'23px / {bin_px:.2f}px = {23/bin_px:.1f}개 (이론)')
print(f'이론 bin 수: {(23/bin_px)**2:.1f}개')


타일 범위: col 2233.0~2256.0, row 659.0~682.0
실제 bin 수: 45

unique col 수: 45
unique row 수: 45
col 간격: [0.01153106 0.01153106 0.01153106 0.01153106]
row 간격: [0.01125548 0.01125548 0.01125548 0.01125548]

8um bin = 2.69px (hires)
23px / 2.69px = 8.6개 (이론)
이론 bin 수: 73.1개


In [2]:

import numpy as np
import pandas as pd
import json

with open('/project_antwerp/hbae/data/visium_hd_tonsil/binned_outputs/square_008um/spatial/scalefactors_json.json') as f:
    sf = json.load(f)
scalef = sf['tissue_hires_scalef']

df = pd.read_parquet('/project_antwerp/hbae/data/visium_hd_tonsil/binned_outputs/square_008um/spatial/tissue_positions.parquet', engine='pyarrow')
df = df[df['in_tissue']==1].reset_index(drop=True)

# fullres 좌표에서 bin 간격 확인
bin_col_full = np.sort(df['pxl_col_in_fullres'].unique())
print(f'fullres bin 간격 (첫 5개): {np.diff(bin_col_full[:6])}')
print(f'fullres bin 간격 평균: {np.diff(bin_col_full).mean():.2f}px')

# hires 좌표에서 bin 간격
bin_col_hi = bin_col_full * scalef
print(f'hires bin 간격 (첫 5개): {np.diff(bin_col_hi[:6])}')
print(f'hires bin 간격 평균: {np.diff(bin_col_hi).mean():.4f}px')

um_per_px = sf['microns_per_pixel'] / sf['tissue_hires_scalef']
print(f'8um = {8/um_per_px:.2f}px (hires 이론값)')
print(f'실제 간격: {np.diff(bin_col_hi).mean():.4f}px')


fullres bin 간격 (첫 5개): [0.12530229 0.12530229 0.12530229 0.1253023  0.1253023 ]
fullres bin 간격 평균: 0.04px
hires bin 간격 (첫 5개): [0.01153106 0.01153106 0.01153106 0.01153106 0.01153106]
hires bin 간격 평균: 0.0032px
8um = 2.69px (hires 이론값)
실제 간격: 0.0032px


In [3]:

import pandas as pd
import json

with open('/project_antwerp/hbae/data/visium_hd_tonsil/binned_outputs/square_008um/spatial/scalefactors_json.json') as f:
    sf = json.load(f)

df = pd.read_parquet('/project_antwerp/hbae/data/visium_hd_tonsil/binned_outputs/square_008um/spatial/tissue_positions.parquet', engine='pyarrow')
df = df[df['in_tissue']==1]

# 실제 값 확인
print('pxl_col_in_fullres 샘플:')
print(df['pxl_col_in_fullres'].head(10).values)
print()
print(f'min: {df.pxl_col_in_fullres.min():.2f}')
print(f'max: {df.pxl_col_in_fullres.max():.2f}')
print(f'범위: {df.pxl_col_in_fullres.max() - df.pxl_col_in_fullres.min():.2f}')
print()

# array_col 확인
print('array_col 샘플:')
print(df['array_col'].head(10).values)
print(f'array_col 간격: {df.array_col.diff().value_counts().head(3)}')


pxl_col_in_fullres 샘플:
[24376.9115614  24406.1384402  24435.36531867 24464.59219679
 24493.81907457 24523.04595201 24552.27282911 24581.49970587
 24610.72658229 24639.95345837]

min: 24272.03
max: 48838.68
범위: 24566.65

array_col 샘플:
[0 1 2 3 4 5 6 7 8 9]
array_col 간격: array_col
1.000000e+00    700714
4.294966e+09       820
3.000000e+00        12
Name: count, dtype: int64


In [4]:

import numpy as np
import pandas as pd
import json

with open('/project_antwerp/hbae/data/visium_hd_tonsil/binned_outputs/square_008um/spatial/scalefactors_json.json') as f:
    sf = json.load(f)
scalef = sf['tissue_hires_scalef']

df = pd.read_parquet('/project_antwerp/hbae/data/visium_hd_tonsil/binned_outputs/square_008um/spatial/tissue_positions.parquet', engine='pyarrow')
df = df[df['in_tissue']==1].reset_index(drop=True)
df['hires_col'] = df['pxl_col_in_fullres'] * scalef
df['hires_row'] = df['pxl_row_in_fullres'] * scalef

bin_col = df['hires_col'].values
bin_row = df['hires_row'].values

col_start, row_start = 2233.0, 659.0
tile_size = 23

in_tile = (
    (bin_col >= col_start) & (bin_col < col_start + tile_size) &
    (bin_row >= row_start) & (bin_row < row_start + tile_size)
)

tile_cols = bin_col[in_tile]
tile_rows = bin_row[in_tile]

print(f'실제 bin 수: {in_tile.sum()}')
print(f'bin hires col 값들: {np.sort(tile_cols)[:8]}')
print(f'bin hires col 간격: {np.diff(np.sort(np.unique(tile_cols)))[:5]}')
print(f'bin hires row 간격: {np.diff(np.sort(np.unique(tile_rows)))[:5]}')
print()
print(f'col 방향 bin 수: {len(np.unique(tile_cols))}')
print(f'row 방향 bin 수: {len(np.unique(tile_rows))}')
print(f'이론: {23/2.69:.1f} × {23/2.69:.1f} = {(23/2.69)**2:.0f}개')


실제 bin 수: 45
bin hires col 값들: [2233.65683068 2233.66836174 2233.6798928  2233.69142387 2233.70295493
 2236.34642992 2236.35796102 2236.36949212]
bin hires col 간격: [0.01153106 0.01153106 0.01153106 0.01153106 2.64347499]
bin hires row 간격: [0.01125548 0.01125548 0.01125548 0.01125548 0.01125548]

col 방향 bin 수: 45
row 방향 bin 수: 45
이론: 8.6 × 8.6 = 73개


In [5]:

import numpy as np
import pandas as pd
import json

with open('/project_antwerp/hbae/data/visium_hd_tonsil/binned_outputs/square_008um/spatial/scalefactors_json.json') as f:
    sf = json.load(f)
scalef = sf['tissue_hires_scalef']

df = pd.read_parquet('/project_antwerp/hbae/data/visium_hd_tonsil/binned_outputs/square_008um/spatial/tissue_positions.parquet', engine='pyarrow')
df = df[df['in_tissue']==1].reset_index(drop=True)

bin_col_full = df['pxl_col_in_fullres'].values
bin_row_full = df['pxl_row_in_fullres'].values

# 첫 20개 fullres 좌표 출력
print('fullres col 첫 20개:')
print(np.sort(bin_col_full)[:20])
print()
print(f'fullres col 간격 (첫 10개): {np.diff(np.sort(bin_col_full))[:10]}')
print()

# array_row, array_col로 정렬해서 확인
df_sorted = df.sort_values(['array_row', 'array_col'])
print('array_row=0 인 bin들:')
row0 = df_sorted[df_sorted['array_row']==0]
print(row0[['array_col', 'pxl_col_in_fullres', 'pxl_row_in_fullres']].head(10).to_string())


fullres col 첫 20개:
[24272.03229827 24272.15760056 24272.28290285 24272.40820514
 24272.53350743 24272.65880973 24272.78411204 24272.90941434
 24273.03471665 24273.16001897 24273.28532129 24273.41062361
 24273.53592593 24273.66122826 24273.78653059 24273.91183293
 24274.03713527 24274.16243761 24274.28773995 24274.4130423 ]

fullres col 간격 (첫 10개): [0.12530229 0.12530229 0.12530229 0.1253023  0.1253023  0.1253023
 0.12530231 0.12530231 0.12530231 0.12530232]

array_row=0 인 bin들:
   array_col  pxl_col_in_fullres  pxl_row_in_fullres
0          0        24376.911561        31731.475716
1          1        24406.138440        31731.353265
2          2        24435.365319        31731.230813
3          3        24464.592197        31731.108362
4          4        24493.819075        31730.985910
5          5        24523.045952        31730.863459
6          6        24552.272829        31730.741007
7          7        24581.499706        31730.618555
8          8        24610.726582        

In [6]:
\
import numpy as np
import pandas as pd
import json

with open('/project_antwerp/hbae/data/visium_hd_tonsil/binned_outputs/square_008um/spatial/scalefactors_json.json') as f:
    sf = json.load(f)
scalef = sf['tissue_hires_scalef']

df = pd.read_parquet('/project_antwerp/hbae/data/visium_hd_tonsil/binned_outputs/square_008um/spatial/tissue_positions.parquet', engine='pyarrow')
df = df[df['in_tissue']==1].reset_index(drop=True)

# array_col, array_row 기반으로 타일 찾기
# 23px = 68um, 8um bin 기준 8.5개 → 8x8 = 64개 또는 9x9 = 81개

# hires에서 bin 중심 좌표 (array 기반)
# 첫 번째 bin의 hires 좌표
df_sorted = df.sort_values(['array_row', 'array_col']).reset_index(drop=True)
df_sorted['hires_col'] = df_sorted['pxl_col_in_fullres'] * scalef
df_sorted['hires_row'] = df_sorted['pxl_row_in_fullres'] * scalef

# bin 간격 (hires)
hi_col_gap = (df_sorted[df_sorted['array_row']==0]['hires_col'].diff().dropna().mean())
hi_row_gap = (df_sorted[df_sorted['array_col']==0]['hires_row'].diff().dropna().abs().mean())

print(f'hires bin 간격 (col): {hi_col_gap:.4f}px')
print(f'hires bin 간격 (row): {hi_row_gap:.4f}px')
print(f'이론: {8/( 0.2737/scalef):.4f}px')
print()
print(f'23px 안에 들어가는 bin 수:')
print(f'  col 방향: {23/hi_col_gap:.1f}개')
print(f'  row 방향: {23/hi_row_gap:.1f}개')
print(f'  전체: {(23/hi_col_gap) * (23/hi_row_gap):.0f}개')
print()

# 타일을 bin 중심 기준으로 찾기
# 첫 번째 타일의 bin 중심들
tile_col_start = df_sorted['hires_col'].min()
tile_row_start = df_sorted['hires_row'].min()

# 타일 크기를 px가 아닌 bin 개수로 정의
n_bins = 8  # 8×8 = 64개 (≈68um)
tile_col_end = tile_col_start + n_bins * hi_col_gap
tile_row_end = tile_row_start + n_bins * hi_row_gap

in_tile = (
    (df_sorted['hires_col'] >= tile_col_start) &
    (df_sorted['hires_col'] < tile_col_end) &
    (df_sorted['hires_row'] >= tile_row_start) &
    (df_sorted['hires_row'] < tile_row_end)
)
print(f'8×8 bin 기준 타일: {in_tile.sum()}개 bin')
print(f'물리 크기: {n_bins * hi_col_gap * (0.2737/scalef):.1f}um × {n_bins * hi_row_gap * (0.2737/scalef):.1f}um')


hires bin 간격 (col): 2.6994px
hires bin 간격 (row): 2.6893px
이론: 2.6898px

23px 안에 들어가는 bin 수:
  col 방향: 8.5개
  row 방향: 8.6개
  전체: 73개

8×8 bin 기준 타일: 45개 bin
물리 크기: 64.2um × 64.0um


In [7]:

import numpy as np
import pandas as pd
import json

with open('/project_antwerp/hbae/data/visium_hd_tonsil/binned_outputs/square_008um/spatial/scalefactors_json.json') as f:
    sf = json.load(f)
scalef = sf['tissue_hires_scalef']

df = pd.read_parquet('/project_antwerp/hbae/data/visium_hd_tonsil/binned_outputs/square_008um/spatial/tissue_positions.parquet', engine='pyarrow')
df = df[df['in_tissue']==1].reset_index(drop=True)

# array_row=0 인 bin들만 (같은 행)
row0 = df[df['array_row']==0].sort_values('array_col').head(10)
print('array_row=0 첫 10개 bin:')
print(row0[['array_col', 'pxl_col_in_fullres', 'pxl_row_in_fullres']].to_string())
print()

# array_col=0 인 bin들만 (같은 열)
col0 = df[df['array_col']==0].sort_values('array_row').head(10)
print('array_col=0 첫 10개 bin:')
print(col0[['array_row', 'pxl_col_in_fullres', 'pxl_row_in_fullres']].to_string())
print()

# 같은 array_row에서 pxl_row_in_fullres가 다른가?
print('array_row=0에서 pxl_row_in_fullres 변화:')
print(f'  min: {row0.pxl_row_in_fullres.min():.2f}')
print(f'  max: {row0.pxl_row_in_fullres.max():.2f}')
print(f'  차이: {row0.pxl_row_in_fullres.max() - row0.pxl_row_in_fullres.min():.2f}px')
print(f'  (10개 bin에 걸쳐 row 좌표가 변하면 기울어진 것)')
print()

# 같은 array_col에서 pxl_col_in_fullres가 다른가?
print('array_col=0에서 pxl_col_in_fullres 변화:')
print(f'  min: {col0.pxl_col_in_fullres.min():.2f}')
print(f'  max: {col0.pxl_col_in_fullres.max():.2f}')
print(f'  차이: {col0.pxl_col_in_fullres.max() - col0.pxl_col_in_fullres.min():.2f}px')


array_row=0 첫 10개 bin:
   array_col  pxl_col_in_fullres  pxl_row_in_fullres
0          0        24376.911561        31731.475716
1          1        24406.138440        31731.353265
2          2        24435.365319        31731.230813
3          3        24464.592197        31731.108362
4          4        24493.819075        31730.985910
5          5        24523.045952        31730.863459
6          6        24552.272829        31730.741007
7          7        24581.499706        31730.618555
8          8        24610.726582        31730.496104
9          9        24639.953458        31730.373652

array_col=0 첫 10개 bin:
      array_row  pxl_col_in_fullres  pxl_row_in_fullres
0             0        24376.911561        31731.475716
827           1        24376.786256        31702.252385
1655          2        24376.660951        31673.029054
2481          3        24376.535646        31643.805725
3304          4        24376.410340        31614.582396
4133          5        24376.28503

In [9]:

import numpy as np
import scanpy as sc

# Visium HD 전체 유전자로 top 20
adata = sc.read_10x_h5('/project_antwerp/hbae/data/visium_hd_tonsil/binned_outputs/square_008um/filtered_feature_bc_matrix.h5')
adata.var_names_make_unique()
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

mean_expr = np.array(adata.X.mean(axis=0)).flatten()
top20_idx = np.argsort(mean_expr)[::-1][:20]
top20_genes = adata.var_names[top20_idx]
print('Visium HD 전체 유전자 top 20:')
for i, (g, v) in enumerate(zip(top20_genes, mean_expr[top20_idx])):
    print(f'  {i+1:2d}. {g}: {v:.3f}')

# HVG 2000개에 있는지 확인
hvg = open('/project_antwerp/hbae/data/0228_HVG_NEW/0228_HVG_Finetune_gene_list_full.txt').read().strip().split('\n')
hvg_set = set(hvg)
print()
print('HVG 2000개 포함 여부:')
for g in top20_genes:
    print(f"  {g}: {'✅' if g in hvg_set else '❌ 없음'}")


/opt/conda/lib/python3.11/site-packages/anndata/_core/anndata.py:1798: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/opt/conda/lib/python3.11/site-packages/anndata/_core/anndata.py:1798: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.
/opt/conda/lib/python3.11/site-packages/legacy_api_wrap/__init__.py:88: UserWarning: Some cells have zero counts
  return fn(*args_all, **kw)


Visium HD 전체 유전자 top 20:
   1. IGKC: 2.070
   2. CD74: 1.759
   3. TMSB4X: 1.340
   4. IGHG1: 0.984
   5. CORO1A: 0.892
   6. TMSB10: 0.772
   7. CLU: 0.754
   8. VIM: 0.738
   9. ACTB: 0.711
  10. IGHA1: 0.695
  11. PFN1: 0.658
  12. EEF2: 0.603
  13. LTB: 0.596
  14. TAPBP: 0.493
  15. CCL19: 0.488
  16. APOE: 0.458
  17. MT-CO3: 0.457
  18. LYZ: 0.443
  19. CST3: 0.433
  20. ZFP36L2: 0.425

HVG 2000개 포함 여부:
  IGKC: ✅
  CD74: ✅
  TMSB4X: ❌ 없음
  IGHG1: ✅
  CORO1A: ✅
  TMSB10: ❌ 없음
  CLU: ✅
  VIM: ✅
  ACTB: ❌ 없음
  IGHA1: ✅
  PFN1: ❌ 없음
  EEF2: ❌ 없음
  LTB: ✅
  TAPBP: ❌ 없음
  CCL19: ✅
  APOE: ✅
  MT-CO3: ❌ 없음
  LYZ: ✅
  CST3: ❌ 없음
  ZFP36L2: ❌ 없음


In [10]:

import numpy as np
import scanpy as sc

adata = sc.read_10x_h5('/project_antwerp/hbae/data/visium_hd_tonsil/binned_outputs/square_008um/filtered_feature_bc_matrix.h5')
adata.var_names_make_unique()
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)

mean_expr = np.array(adata.X.mean(axis=0)).flatten()
top300_all = set(adata.var_names[np.argsort(mean_expr)[::-1][:300]])

hvg = set(open('/project_antwerp/hbae/data/0228_HVG_NEW/0228_HVG_Finetune_gene_list_full.txt').read().strip().split('\n'))

overlap = top300_all & hvg
print(f'전체 유전자 top 300 중 HVG에 있는 유전자: {len(overlap)}개')
print(f'HVG에 없는 유전자: {300 - len(overlap)}개')

# 겹치는 유전자 top 20
top300_list = list(adata.var_names[np.argsort(mean_expr)[::-1][:300]])
overlap_ordered = [g for g in top300_list if g in hvg]
print()
print('겹치는 유전자 top 20:')
for i, g in enumerate(overlap_ordered[:20]):
    idx = list(adata.var_names).index(g)
    print(f'  {i+1:2d}. {g}: {mean_expr[idx]:.3f}')


/opt/conda/lib/python3.11/site-packages/anndata/_core/anndata.py:1798: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/opt/conda/lib/python3.11/site-packages/anndata/_core/anndata.py:1798: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/opt/conda/lib/python3.11/site-packages/legacy_api_wrap/__init__.py:88: UserWarning: Some cells have zero counts
  return fn(*args_all, **kw)


전체 유전자 top 300 중 HVG에 있는 유전자: 116개
HVG에 없는 유전자: 184개

겹치는 유전자 top 20:
   1. IGKC: 2.070
   2. CD74: 1.759
   3. IGHG1: 0.984
   4. CORO1A: 0.892
   5. CLU: 0.754
   6. VIM: 0.738
   7. IGHA1: 0.695
   8. LTB: 0.596
   9. CCL19: 0.488
  10. APOE: 0.458
  11. LYZ: 0.443
  12. CD37: 0.422
  13. CXCR4: 0.418
  14. S100A9: 0.407
  15. CCL21: 0.403
  16. CYBA: 0.398
  17. H3F3A: 0.396
  18. LSP1: 0.385
  19. CD44: 0.370
  20. PTGDS: 0.368


In [11]:

import numpy as np
import torch
import torch.nn.functional as F
from scipy.stats import pearsonr
import scanpy as sc

# 전체 유전자로 Visium HD top 300 선택
adata = sc.read_10x_h5('/project_antwerp/hbae/data/visium_hd_tonsil/binned_outputs/square_008um/filtered_feature_bc_matrix.h5')
adata.var_names_make_unique()
sc.pp.normalize_total(adata)
sc.pp.log1p(adata)
mean_expr_all = np.array(adata.X.mean(axis=0)).flatten()
top300_all = list(adata.var_names[np.argsort(mean_expr_all)[::-1][:300]])

# HVG와 겹치는 유전자만
hvg = open('/project_antwerp/hbae/data/0228_HVG_NEW/0228_HVG_Finetune_gene_list_full.txt').read().strip().split('\n')
hvg_set = set(hvg)
overlap_genes = [g for g in top300_all if g in hvg_set]
print(f'평가 유전자: {len(overlap_genes)}개 (전체 top300 중 HVG 포함)')

# HVG index 찾기
gene_to_hvg_idx = {g: i for i, g in enumerate(hvg)}
eval_idx = [gene_to_hvg_idx[g] for g in overlap_genes]

# 임베딩 로드
train_embs  = torch.tensor(np.load('/project_antwerp/hbae/Loki_output/0228_epoch10_finetune_embedding/fold_01/train_text_embs.npy')).float()
train_exprs = torch.tensor(np.load('/project_antwerp/hbae/Loki_output/0228_epoch10_finetune_embedding/fold_01/train_exprs.npy')).float()
val_embs    = torch.tensor(np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_01/tile_img_embs.npy')).float()
val_exprs   = np.load('/project_antwerp/hbae/Loki_output/visium_hd_embeddings/fold_01/tile_exprs.npy')

train_norm = F.normalize(train_embs, dim=-1)
val_norm   = F.normalize(val_embs, dim=-1)

# 예측
preds = []
for i in range(len(val_norm)):
    sim = val_norm[i] @ train_norm.T
    idx = sim.topk(500).indices
    w   = sim[idx] / sim[idx].sum()
    preds.append((w[:, None] * train_exprs[idx]).sum(0).numpy())
preds = np.array(preds)

# 평가 (전체 top300 기준 116개)
preds_eval   = preds[:, eval_idx]
val_eval     = val_exprs[:, eval_idx]

spot_corrs, gene_corrs = [], []
for i in range(len(preds_eval)):
    if val_eval[i].std() > 1e-8:
        r, _ = pearsonr(preds_eval[i], val_eval[i])
        if np.isfinite(r): spot_corrs.append(r)
for g in range(preds_eval.shape[1]):
    if val_eval[:, g].std() > 1e-8:
        r, _ = pearsonr(preds_eval[:, g], val_eval[:, g])
        if np.isfinite(r): gene_corrs.append(r)

print()
print('='*55)
print('평가 방식: 전체 유전자 top300 중 HVG 있는 116개')
print('='*55)
print(f'Spot-wise PCC: mean={np.mean(spot_corrs):.4f}')
print(f'Gene-wise PCC: mean={np.mean(gene_corrs):.4f}')
print()
print('비교 (기존 HVG top300):')
print('  Spot-wise PCC: 0.368')
print('  Gene-wise PCC: 0.040')


/opt/conda/lib/python3.11/site-packages/anndata/_core/anndata.py:1798: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/opt/conda/lib/python3.11/site-packages/anndata/_core/anndata.py:1798: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/opt/conda/lib/python3.11/site-packages/legacy_api_wrap/__init__.py:88: UserWarning: Some cells have zero counts
  return fn(*args_all, **kw)


평가 유전자: 116개 (전체 top300 중 HVG 포함)

평가 방식: 전체 유전자 top300 중 HVG 있는 116개
Spot-wise PCC: mean=0.3227
Gene-wise PCC: mean=0.0390

비교 (기존 HVG top300):
  Spot-wise PCC: 0.368
  Gene-wise PCC: 0.040


In [13]:

import numpy as np

exprs = np.load('/project_antwerp/hbae/Loki_output/visium_hd_array_embeddings/fold_01/tile_exprs.npy')
print(f'shape: {exprs.shape}')
print(f'mean: {exprs.mean():.4f}')
print(f'std: {exprs.std():.4f}')
print(f'zeros: {(exprs==0).mean():.4f}')
print(f'max: {exprs.max():.4f}')


shape: (8649, 2000)
mean: 0.8535
std: 1.1009
zeros: 0.4903
max: 8.9609


In [14]:

import numpy as np
import pandas as pd

scalef = 0.09202595
df = pd.read_parquet('/project_antwerp/hbae/data/visium_hd_tonsil/binned_outputs/square_008um/spatial/tissue_positions.parquet', engine='pyarrow')
df = df[df['in_tissue']==1].reset_index(drop=True)

coords = np.load('/project_antwerp/hbae/Loki_output/visium_hd_array_embeddings/fold_01/tile_coords.npy')
print(f'타일 수: {len(coords)}')

# 첫 5개 타일의 array 기반 bin 수 확인
# coords는 img_col_start, img_row_start
# 타일 하나당 이론 bin 수: 9×9 = 81개인지 확인
n = 9
bin_counts = []
for col_start, row_start in coords[:20]:
    # img 좌표에서 가장 가까운 bin 찾기
    df['hires_col'] = df['pxl_col_in_fullres'] * scalef
    df['hires_row'] = df['pxl_row_in_fullres'] * scalef
    in_tile = (
        (df['hires_col'] >= col_start) & (df['hires_col'] < col_start + 23) &
        (df['hires_row'] >= row_start) & (df['hires_row'] < row_start + 23)
    )
    bin_counts.append(in_tile.sum())

bin_counts = np.array(bin_counts)
print(f'타일당 bin 수 (첫 20개): mean={bin_counts.mean():.1f}, min={bin_counts.min()}, max={bin_counts.max()}')


타일 수: 8649
타일당 bin 수 (첫 20개): mean=76.3, min=71, max=81
